# BioVis · Q3/Q4 Data Transformation

**Unit: one US county/county equivalent.** Q3 joins chronic-disease prevalence to overall SVI and highlights NYC; Q4 reuses that snapshot for four themes × five conditions. No plots or sketches are produced.

**Execution evidence:** saved code-cell outputs show the actual run state. The delivered executed notebook uses the two user-supplied CDC CSVs; source hashes, raw counts, merged counts, complete tables and validation appear below and in the exported manifest. Regenerating this notebook with the maintenance builder clears downstream outputs until it is run again. No Q1 CSV was supplied or required.

## 1. Data Loading

Run in **Google Colab** (pandas is preinstalled). No local environment or repository clone is required. All pipeline functions are included below. For GitHub review, save the executed notebook from Colab after the run; the saved outputs will show the actual DataFrames.

**Obtain the two original CSVs:**

1. [PLACES 2025 county GIS-friendly](https://data.cdc.gov/d/i46a-9kgh): export the full CSV, retaining adjusted estimates and CI strings.
2. [SVI official selector](https://svi.cdc.gov/dataDownloads/data-download.html): **2022 → United States → Counties → CSV**. Use the original official county export; never a tract file or state-specific rankings. If the official selector no longer offers the download, obtain the original official file from the teammate. Stop if unavailable.
3. Optionally provide `q1_county_prevalence.csv` for safe reuse/scope comparison.

Choose `INPUT_MODE = 'upload'` below for Colab's file picker; upload both original CDC files together, optionally including Q1. The code recognizes the datasets by their actual headers and caches them with project-standard names. `INPUT_MODE = 'download'` fetches PLACES and needs an exact official `SVI_CSV_URL` obtained from the selector. `INPUT_MODE = 'cached'` uses files already in `data/raw/`. Direct SVI endpoints are deliberately not guessed.

Raw files are never silently overwritten. An uploaded file's schema cannot prove its release year or national ranking universe; confirm these in the CDC download selector and retain the original export.


In [18]:
from pathlib import Path
import importlib.util
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
INPUT_MODE = "upload" if IN_COLAB else "cached"  # upload | download | cached
SVI_CSV_URL = None  # exact official US COUNTY CSV URL, if available

if IN_COLAB:
    PROJECT_ROOT = Path("/content/BioVis")
else:
    here = Path.cwd()
    PROJECT_ROOT = here.parent if here.name == "notebooks" else (here / "BioVis" if (here / "BioVis").is_dir() else here)
for folder in ["data/raw", "data/processed", "docs/validation", "src", "notebooks"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Runtime:", "Google Colab" if IN_COLAB else "Local Python")
print("Python:", sys.version.split()[0])
print("pandas installed:", importlib.util.find_spec("pandas") is not None)
print("Input mode:", INPUT_MODE)
for name in ["places_2025_county_gis.csv", "svi_2022_us_county.csv"]:
    print(name, "present" if (PROJECT_ROOT / "data/raw" / name).exists() else "MISSING")
print("Optional Q1:", "present" if (PROJECT_ROOT / "data/processed/q1_county_prevalence.csv").exists() else "not supplied")
print("No empirical transformation is claimed by this preflight.")


Runtime: Local Python
Python: 3.13.7
pandas installed: True
Input mode: cached
places_2025_county_gis.csv present
svi_2022_us_county.csv present
Optional Q1: not supplied
No empirical transformation is claimed by this preflight.


The following helpers inspect schema, preserve source strings and cache full official CSVs. They are copied from the standalone `src/q3_q4_transform.py`; no `pip install` is necessary in Colab.


In [19]:
"""CDC county data transformation only; Python 3.10+, pandas 2+.

Run from the repository root: python src/q3_q4_transform.py --root .
The notebook contains the same implementation for standalone Colab execution.
No plots, imputation, population weighting, or tract-rank aggregation.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import re
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PLACES_ID = "i46a-9kgh"
PLACES_URL = f"https://data.cdc.gov/api/views/{PLACES_ID}/rows.csv?accessType=DOWNLOAD"
SVI_PORTAL = "https://svi.cdc.gov/dataDownloads/data-download.html"
DISEASES = ["DIABETES", "CHD", "BPHIGH", "OBESITY", "COPD"]
THEMES = {
    "RPL_THEME1": "Socioeconomic Status",
    "RPL_THEME2": "Household Characteristics",
    "RPL_THEME3": "Racial and Ethnic Minority Status",
    "RPL_THEME4": "Housing Type and Transportation",
}
RANKS = ["RPL_THEMES", *THEMES]
NYC = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan",
       "36081": "Queens", "36085": "Staten Island"}
SAMPLE_FIPS = [*NYC, "06037", "17031", "48201", "04013", "12086", "28049", "01001"]
US_STATES = set("01 02 04 05 06 08 09 10 11 12 13 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 44 45 46 47 48 49 50 51 53 54 55 56".split())
MISSING_TOKENS = {"", "na", "n/a", "null", "none", "nan", "suppressed", "*", "**", "--", "-", "—"}
RAW_NAMES = {"places": "places_2025_county_gis.csv", "svi": "svi_2022_us_county.csv"}
VALUE_COLS = [f"{d}_adj" for d in DISEASES]
HEALTH_COLS = [f"{d}_adj{s}" for d in DISEASES for s in ("", "_lo", "_hi")]

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)


In [20]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_raw(path):
    """Read strings before cleaning so leading zeros and source missing tokens survive."""
    df = pd.read_csv(path, dtype="string", keep_default_na=False, encoding="utf-8-sig")
    original = list(df.columns)
    df.columns = df.columns.str.strip()
    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate column names after trimming in {path}: {original}")
    if df.empty:
        raise ValueError(f"No data rows in {path}")
    return df


def col(df, *candidates, required=True):
    lookup = {str(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lookup:
            return lookup[candidate.casefold()]
    if required:
        raise ValueError(f"Missing required column {candidates}; actual columns: {list(df.columns)}")
    return None


def inspect_frame(df, dataset):
    """Report as-read dtypes separately from inferred numeric content."""
    records = []
    for name in df:
        s = df[name].astype("string").str.strip()
        numeric = pd.to_numeric(s, errors="coerce")
        records.append({"dataset": dataset, "column": name, "read_dtype": str(df[name].dtype),
                        "source_blank_count": int(s.eq("").sum()),
                        "numeric_parseable_count": int(numeric.notna().sum()),
                        "distinct_values": int(s.nunique(dropna=False))})
    return pd.DataFrame(records)


def download_csv(url, dest):
    """Atomically cache a complete CSV; never replace an existing raw snapshot."""
    dest = Path(dest)
    if dest.exists():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "BioVis-academic-transformation/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        payload = response.read()
        final_url = response.geturl()
    # Parse before writing: CDC block pages must not become a raw-data cache.
    import io
    probe = pd.read_csv(io.BytesIO(payload), dtype="string", nrows=2)
    if len(probe.columns) < 5 or probe.empty:
        raise ValueError(f"The URL did not return a nonempty tabular CDC CSV: {url}")
    temp = dest.with_suffix(".download")
    temp.write_bytes(payload)
    temp.replace(dest)
    source_record = dest.parents[2] / "docs/validation/sources" / dest.with_suffix(".source.json").name
    json_write(source_record, {
        "requested_url": url, "resolved_url": final_url, "retrieved_utc": utc_now(),
        "sha256": file_sha256(dest), "bytes": dest.stat().st_size,
    })


def load_sources(root, download=True, svi_url=None):
    """Inspect the optional teammate file first. Never modify it."""
    root = Path(root)
    q1_path = root / "data/processed/q1_county_prevalence.csv"
    q1 = read_raw(q1_path) if q1_path.exists() else None
    frames, provenance, failures = {}, {}, []
    for dataset, filename in RAW_NAMES.items():
        path = root / "data/raw" / filename
        url = PLACES_URL if dataset == "places" else svi_url
        if not path.exists() and download and url:
            try:
                download_csv(url, path)
            except Exception as exc:
                failures.append(f"{dataset} download: {type(exc).__name__}: {exc}")
        if not path.exists():
            failures.append(f"Missing {path}. " + (
                f"Download CDC PLACES 2025 County GIS-Friendly ({PLACES_ID})." if dataset == "places" else
                f"At {SVI_PORTAL} choose 2022 / United States / Counties / CSV; upload the national county file, not tracts or individual-state rankings."))
            continue
        frames[dataset] = read_raw(path)
        sidecar = root / "docs/validation/sources" / path.with_suffix(".source.json").name
        provenance[dataset] = {
            "path": str(path.relative_to(root)), "sha256": file_sha256(path),
            "bytes": path.stat().st_size, "rows": len(frames[dataset]),
            "columns": list(frames[dataset].columns), "observed_utc": utc_now(),
            "download_record": json.loads(sidecar.read_text()) if sidecar.exists() else None,
            "source_identity": (f"Expected official PLACES 2025 county GIS CSV ({PLACES_ID})" if dataset == "places" else
                                "Expected official SVI 2022 US county CSV, national rankings"),
            "identity_note": "Schema checks alone cannot authenticate an uploaded file's year or ranking universe; retain the original CDC download and source record.",
        }
    if failures:
        raise FileNotFoundError("\n".join(failures))
    inspections = [inspect_frame(df, name) for name, df in frames.items()]
    if q1 is not None:
        inspections.append(inspect_frame(q1, "q1_candidate"))
        provenance["q1_candidate"] = {"path": str(q1_path.relative_to(root)), "sha256": file_sha256(q1_path), "rows": len(q1)}
    raw_summary = pd.DataFrame([
        {"dataset": name, "rows": len(df), "columns": len(df.columns),
         "duplicate_full_rows": int(df.duplicated().sum())} for name, df in frames.items()
    ])
    return frames, q1, provenance, pd.concat(inspections, ignore_index=True), raw_summary


In [4]:
# Optional Colab upload: identify the datasets using actual headers.
if INPUT_MODE not in {"upload", "download", "cached"}:
    raise ValueError("INPUT_MODE must be upload, download or cached")
if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Upload mode uses Google Colab; locally use cached or download mode")
    from google.colab import files
    import io
    uploaded = files.upload()  # choose both original CSVs and optionally the Q1 CSV
    destinations = set()
    for original_name, payload in uploaded.items():
        header = pd.read_csv(io.BytesIO(payload), nrows=1, dtype="string").columns
        names = {str(c).strip().casefold() for c in header}
        if Path(original_name).name.casefold() == "q1_county_prevalence.csv":
            dest = PROJECT_ROOT / "data/processed/q1_county_prevalence.csv"
        elif {"countyfips", "diabetes_adjprev", "diabetes_adj95ci"}.issubset(names):
            dest = PROJECT_ROOT / "data/raw" / RAW_NAMES["places"]
        elif {"rpl_themes", "rpl_theme1", "rpl_theme2", "rpl_theme3", "rpl_theme4"}.issubset(names) and ({"fips", "stcnty"} & names):
            dest = PROJECT_ROOT / "data/raw" / RAW_NAMES["svi"]
        else:
            raise ValueError(f"Cannot identify {original_name}; inspect its actual headers: {list(header)}")
        if dest in destinations:
            raise ValueError(f"Multiple uploads map to {dest.name}; upload exactly one of each dataset")
        destinations.add(dest)
        if dest.exists() and dest.read_bytes() != payload:
            raise FileExistsError(f"A different cached file already exists at {dest}; choose a fresh project folder or explicitly manage that snapshot")
        dest.write_bytes(payload)
        print("Cached:", original_name, "->", dest.relative_to(PROJECT_ROOT))

try:
    frames, q1, provenance, inspection, raw_summary = load_sources(
        PROJECT_ROOT, download=INPUT_MODE == "download", svi_url=SVI_CSV_URL)
except Exception as exc:
    json_write(PROJECT_ROOT / "docs/validation/last_attempt.json", {
        "status": "blocked_or_failed", "observed_utc": utc_now(),
        "error_type": type(exc).__name__, "error": str(exc),
        "note": "No new empirical results were generated by this attempt."})
    raise

from IPython.display import display
display(raw_summary)
display(inspection)  # actual field names, read dtypes and numeric-parseability counts
for dataset, raw in frames.items():
    print(dataset, "actual columns:", list(raw.columns))
    display(raw.head())
if q1 is not None:
    print("Q1 candidate inspected; source file will not be overwritten")
    display(q1.head())


places actual columns: ['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TotalPopulation', 'TotalPop18plus', 'ACCESS2_CrudePrev', 'ACCESS2_Crude95CI', 'ACCESS2_AdjPrev', 'ACCESS2_Adj95CI', 'ARTHRITIS_CrudePrev', 'ARTHRITIS_Crude95CI', 'ARTHRITIS_AdjPrev', 'ARTHRITIS_Adj95CI', 'BINGE_CrudePrev', 'BINGE_Crude95CI', 'BINGE_AdjPrev', 'BINGE_Adj95CI', 'BPHIGH_CrudePrev', 'BPHIGH_Crude95CI', 'BPHIGH_AdjPrev', 'BPHIGH_Adj95CI', 'BPMED_CrudePrev', 'BPMED_Crude95CI', 'BPMED_AdjPrev', 'BPMED_Adj95CI', 'CANCER_CrudePrev', 'CANCER_Crude95CI', 'CANCER_AdjPrev', 'CANCER_Adj95CI', 'CASTHMA_CrudePrev', 'CASTHMA_Crude95CI', 'CASTHMA_AdjPrev', 'CASTHMA_Adj95CI', 'CHD_CrudePrev', 'CHD_Crude95CI', 'CHD_AdjPrev', 'CHD_Adj95CI', 'CHECKUP_CrudePrev', 'CHECKUP_Crude95CI', 'CHECKUP_AdjPrev', 'CHECKUP_Adj95CI', 'CHOLSCREEN_CrudePrev', 'CHOLSCREEN_Crude95CI', 'CHOLSCREEN_AdjPrev', 'CHOLSCREEN_Adj95CI', 'COLON_SCREEN_CrudePrev', 'COLON_SCREEN_Crude95CI', 'COLON_SCREEN_AdjPrev', 'COLON_SCREEN_Adj95CI', 'COPD

,dataset,rows,columns,duplicate_full_rows
0,places,3143,167,0
1,svi,3144,158,0


,dataset,column,read_dtype,source_blank_count,numeric_parseable_count,distinct_values
0,places,StateAbbr,string,0,0,51
1,places,StateDesc,string,0,0,51
2,places,CountyName,string,0,0,1837
3,places,CountyFIPS,string,0,3143,3143
4,places,TotalPopulation,string,0,35,3083
...,...,...,...,...,...,...
320,svi,MP_NHPI,string,0,3144,57
321,svi,EP_TWOMORE,string,0,3144,129
322,svi,MP_TWOMORE,string,0,3144,62
323,svi,EP_OTHERRACE,string,0,3144,29


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,ACCESS2_CrudePrev,ACCESS2_Crude95CI,ACCESS2_AdjPrev,ACCESS2_Adj95CI,ARTHRITIS_CrudePrev,ARTHRITIS_Crude95CI,ARTHRITIS_AdjPrev,ARTHRITIS_Adj95CI,BINGE_CrudePrev,BINGE_Crude95CI,BINGE_AdjPrev,BINGE_Adj95CI,BPHIGH_CrudePrev,BPHIGH_Crude95CI,...,FOODINSECU_Crude95CI,FOODINSECU_AdjPrev,FOODINSECU_Adj95CI,HOUSINSECU_CrudePrev,HOUSINSECU_Crude95CI,HOUSINSECU_AdjPrev,HOUSINSECU_Adj95CI,SHUTUTILITY_CrudePrev,SHUTUTILITY_Crude95CI,SHUTUTILITY_AdjPrev,SHUTUTILITY_Adj95CI,LACKTRPT_CrudePrev,LACKTRPT_Crude95CI,LACKTRPT_AdjPrev,LACKTRPT_Adj95CI,EMOTIONSPT_CrudePrev,EMOTIONSPT_Crude95CI,EMOTIONSPT_AdjPrev,EMOTIONSPT_Adj95CI,Geolocation
0,KY,Kentucky,Trigg,21221,"14,369","11,280",,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,POINT (-87.8732998555619 36.8063370344073)
1,KY,Kentucky,Wayne,21231,"19,580","15,608",,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,POINT (-84.8283531236799 36.8008845409014)
2,KY,Kentucky,Washington,21229,"12,267","9,420",,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,POINT (-85.175051728938 37.7532982899311)
3,PA,Pennsylvania,Perry,42099,"46,083","36,495",,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,POINT (-77.2622712474235 40.3984083146619)
4,PA,Pennsylvania,Venango,42121,"49,431","39,983",,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,POINT (-79.7581868804174 41.4006288227586)


,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,E_HU,M_HU,E_HH,M_HH,E_POV150,M_POV150,E_UNEMP,M_UNEMP,E_HBURD,M_HBURD,...,E_TWOMORE,M_TWOMORE,E_OTHERRACE,M_OTHERRACE,EP_NOINT,MP_NOINT,EP_AFAM,MP_AFAM,EP_HISP,MP_HISP,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,01,Alabama,AL,01001,Autauga County,01001,"Autauga County, Alabama",594.454786355022,58761,0,24457,41,22308,369,11780,1363,752,215,4389,582,...,1930,598,141,183,10.9,1.7,19.6,0.9,3.2,0.0,1.1,0.4,0.1,0.1,0.0,0.1,3.3,1.0,0.2,0.3
1,01,Alabama,AL,01003,Baldwin County,01003,"Baldwin County, Alabama",1589.861816637086,233420,0,125113,85,90802,1229,42088,2875,3825,609,21039,1527,...,7159,918,992,807,10.9,1.1,8.3,0.3,4.8,0.0,0.9,0.1,0.2,0.1,0.0,0.1,3.1,0.4,0.4,0.3
2,01,Alabama,AL,01005,Barbour County,01005,"Barbour County, Alabama",885.007618621402,24877,0,11673,52,9016,304,8248,594,516,162,2260,321,...,447,174,290,201,31.8,3.3,46.9,0.9,4.8,0.0,0.5,0.1,0.3,0.1,0.0,0.1,1.8,0.7,1.2,0.8
3,01,Alabama,AL,01007,Bibb County,01007,"Bibb County, Alabama",622.469286466726,22251,0,9046,97,7216,252,6047,940,786,269,1671,336,...,382,220,17,30,20.2,4.3,20.7,1.3,2.9,0.0,0.3,0.4,0.1,0.1,0.0,0.2,1.7,1.0,0.1,0.1
4,01,Alabama,AL,01009,Blount County,01009,"Blount County, Alabama",644.890376325486,59077,0,24677,50,21626,386,13348,1075,1578,354,4186,467,...,1634,388,87,80,16.9,2.1,1.2,0.3,9.7,0.0,0.2,0.2,0.1,0.1,0.2,0.2,2.8,0.7,0.1,0.1


## 2. Data Cleaning

Use Q1's documented fields: `pop18plus`, `{m}_adj`, `{m}_adj_lo`, `{m}_adj_hi`. County FIPS is text padded to five digits. Duplicate/malformed identifiers stop execution; 11-digit tract files are rejected. Source blanks and `-999` become NA, never zero. Other invalid values are audited and masked. CI strings are parsed and checked against point estimates. Keep original raw files and the cell-level cleaning audit. Only explicitly out-of-scope territories are separated, with reasons; missing estimates do not remove counties.


In [21]:
def normalize_fips(series, label):
    s = series.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
    bad = ~s.str.fullmatch(r"\d{4,5}", na=False)
    if bad.any():
        examples = series[bad].head(10).tolist()
        raise ValueError(f"{label}: invalid county FIPS ({int(bad.sum())} rows): {examples}. "
                         "County identifiers must have 4 or 5 digits before zero-padding; 11-digit tracts are rejected.")
    s = s.str.zfill(5)
    if s.str.endswith("000").any():
        raise ValueError(f"{label}: found state/national aggregate code ending in 000")
    if s.duplicated().any():
        raise ValueError(f"{label}: duplicate county FIPS; no automatic deduplication: {s[s.duplicated(False)].tolist()[:20]}")
    return s


def text_values(s):
    s = s.astype("string").str.strip()
    return s.mask(s.str.casefold().isin(MISSING_TOKENS))


def numeric_clean(s, fips, dataset, field, issues, lo=0, hi=None, integer=False, record_missing=True):
    raw = s.astype("string").str.strip()
    missing = raw.isna() | raw.str.casefold().isin(MISSING_TOKENS)
    numeric_text = raw.str.replace(r"(?<=\d),(?=\d{3}(?:\D|$))", "", regex=True)
    num = pd.to_numeric(numeric_text.mask(missing), errors="coerce").astype("Float64")
    sentinel = num.eq(-999).fillna(False)
    parse_error = ~missing & num.isna()
    invalid = num.notna() & ~sentinel & (~num.ge(lo) | (num.gt(hi) if hi is not None else False))
    if integer:
        invalid = invalid | (num.notna() & ~sentinel & num.mod(1).ne(0))
    for reason, mask in [("source_missing", missing), ("sentinel_-999", sentinel),
                         ("numeric_parse_error", parse_error), ("out_of_range_or_noninteger", invalid)]:
        if reason == "source_missing" and not record_missing:
            continue
        for idx in s.index[mask.fillna(False)]:
            issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": field,
                           "raw_value": None if pd.isna(raw.loc[idx]) else str(raw.loc[idx]), "reason": reason})
    return num.mask(missing | sentinel | parse_error | invalid)


def parse_ci(s, fips, dataset, disease, issues):
    raw = s.astype("string").str.strip()
    # CDC GIS-friendly intervals are strings such as (8.2, 9.3).
    pattern = r"^\s*[\[(]?\s*(-?\d+(?:\.\d+)?)\s*[,–-]\s*(-?\d+(?:\.\d+)?)\s*[\])]?\s*$"
    extracted = raw.str.extract(pattern)
    missing = raw.isna() | raw.str.casefold().isin(MISSING_TOKENS) | raw.isin(["-999", "-999.0"])
    malformed = ~missing & extracted[0].isna()
    for idx in raw.index[malformed]:
        issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": f"{disease}_adj_ci",
                       "raw_value": str(raw.loc[idx]), "reason": "malformed_ci"})
    # Keep one record of the original interval for every missing/invalid CI.
    for idx in raw.index[missing]:
        issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": f"{disease}_adj_ci",
                       "raw_value": None if pd.isna(raw.loc[idx]) else str(raw.loc[idx]), "reason": "source_missing_or_sentinel_ci"})
    bounds = []
    for i, suffix in enumerate(["lo", "hi"]):
        bounds.append(numeric_clean(extracted[i], fips, dataset, f"{disease}_adj_{suffix}", issues, hi=100, record_missing=False))
    return bounds


def clean_sources(frames):
    issues, exclusions = [], []
    p, s = frames["places"], frames["svi"]
    pfips = normalize_fips(p[col(p, "CountyFIPS")], "PLACES")
    sfips = normalize_fips(s[col(s, "FIPS", "STCNTY")], "SVI")
    # If the file includes both, they must identify the same counties, never tracts.
    if col(s, "STCNTY", required=False) and col(s, "FIPS", required=False):
        stcnty = normalize_fips(s[col(s, "STCNTY")], "SVI STCNTY")
        if not sfips.equals(stcnty):
            raise ValueError("SVI FIPS and STCNTY disagree; confirm a county CSV was supplied")
    if pfips.str[:2].nunique() < 40 or sfips.str[:2].nunique() < 40:
        raise ValueError("Expected national county CSVs, but fewer than 40 state FIPS prefixes were found")
    places = pd.DataFrame({"county_fips": pfips,
                           "state": text_values(p[col(p, "StateAbbr")]),
                           "county": text_values(p[col(p, "CountyName")])})
    adult_col = col(p, "TotalPop18plus", required=False)
    places["pop18plus"] = (numeric_clean(p[adult_col], pfips, "places", "pop18plus", issues, integer=True)
                                   if adult_col else pd.Series(pd.NA, index=p.index, dtype="Float64"))
    for disease in DISEASES:
        places[f"{disease}_adj"] = numeric_clean(p[col(p, f"{disease}_AdjPrev")], pfips, "places", f"{disease}_adj", issues, hi=100)
        lower, upper = parse_ci(p[col(p, f"{disease}_Adj95CI")], pfips, "places", disease, issues)
        # Inconsistent uncertainty bounds are missing, without discarding the estimate/county.
        point = places[f"{disease}_adj"]
        bad_order = (lower.notna() & upper.notna() & lower.gt(upper))
        outside = point.notna() & ((lower.notna() & point.lt(lower)) | (upper.notna() & point.gt(upper)))
        bad = (bad_order | outside).fillna(False)
        for idx in p.index[bad]:
            issues.append({"dataset": "places", "county_fips": str(pfips.loc[idx]), "field": f"{disease}_adj_ci",
                           "raw_value": str(p.loc[idx, col(p, f"{disease}_Adj95CI")]), "reason": "ci_order_or_estimate_outside_ci"})
        places[f"{disease}_adj_lo"] = lower.mask(bad)
        places[f"{disease}_adj_hi"] = upper.mask(bad)
    svi = pd.DataFrame({"county_fips": sfips,
                        "svi_state": text_values(s[col(s, "ST_ABBR")]),
                        "svi_county": text_values(s[col(s, "COUNTY")])})
    for rank in RANKS:
        svi[rank] = numeric_clean(s[col(s, rank)], sfips, "svi", rank, issues, hi=1)
    cleaned = {}
    for label, df in [("places", places), ("svi", svi)]:
        in_scope = df.county_fips.str[:2].isin(US_STATES)
        for fips in df.loc[~in_scope, "county_fips"]:
            exclusions.append({"dataset": label, "county_fips": fips, "reason": "Outside 50 states + DC; not in US national SVI ranking universe"})
        cleaned[label] = df.loc[in_scope].copy()
    return cleaned, pd.DataFrame(issues, columns=["dataset", "county_fips", "field", "raw_value", "reason"]), pd.DataFrame(exclusions, columns=["dataset", "county_fips", "reason"])


In [6]:
cleaned, issues, exclusions = clean_sources(frames)
display(cleaned["places"].head())
display(cleaned["svi"].head())
display(pd.DataFrame([{"dataset": k, "in_scope_counties": len(v), "duplicate_fips": int(v.county_fips.duplicated().sum())} for k, v in cleaned.items()]))
display(issues.groupby(["dataset", "reason"], dropna=False).size().rename("n_cells").reset_index())
display(issues.head(20))
display(exclusions)


,county_fips,state,county,pop18plus,DIABETES_adj,DIABETES_adj_lo,DIABETES_adj_hi,CHD_adj,CHD_adj_lo,CHD_adj_hi,BPHIGH_adj,BPHIGH_adj_lo,BPHIGH_adj_hi,OBESITY_adj,OBESITY_adj_lo,OBESITY_adj_hi,COPD_adj,COPD_adj_lo,COPD_adj_hi
0,21221,KY,Trigg,11280.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,21231,KY,Wayne,15608.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,21229,KY,Washington,9420.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,42099,PA,Perry,36495.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,42121,PA,Venango,39983.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


,county_fips,svi_state,svi_county,RPL_THEMES,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4
0,01001,AL,Autauga County,0.2663,0.2367,0.3675,0.6354,0.2358
1,01003,AL,Baldwin County,0.3487,0.3433,0.2883,0.489,0.4308
2,01005,AL,Barbour County,0.9927,0.9399,0.9399,0.8941,0.9959
3,01007,AL,Bibb County,0.8451,0.8893,0.4954,0.6185,0.8931
4,01009,AL,Blount County,0.6166,0.6866,0.7884,0.413,0.3532


,dataset,in_scope_counties,duplicate_fips
0,places,3143,0
1,svi,3144,0


,dataset,reason,n_cells
0,places,source_missing,935
1,places,source_missing_or_sentinel_ci,935


,dataset,county_fips,field,raw_value,reason
0,places,21221,DIABETES_adj,,source_missing
1,places,21231,DIABETES_adj,,source_missing
2,places,21229,DIABETES_adj,,source_missing
3,places,42099,DIABETES_adj,,source_missing
4,places,42121,DIABETES_adj,,source_missing
5,places,21083,DIABETES_adj,,source_missing
6,places,21059,DIABETES_adj,,source_missing
7,places,42035,DIABETES_adj,,source_missing
8,places,21073,DIABETES_adj,,source_missing
9,places,21177,DIABETES_adj,,source_missing


,dataset,county_fips,reason


## 3. County-Level Merging

Reuse Q1's adjusted values/CIs only when they match the official PLACES release and FIPS universe, including missingness. Otherwise use the independently processed official data and document the mismatch. Never edit the teammate file.

An outer one-to-one join retains every in-scope source county, including unmatched codes and missing values. `join_status` distinguishes source coverage from complete observations. SVI-only county equivalents may extend the Q1 row universe; examine all unmatched records before claiming geography compatibility. No geographic crosswalk or averaging of rankings is applied.


In [22]:
def check_q1(q1, places):
    """Reuse only verified identical values; record scope differences without changing files."""
    report = {"exists": q1 is not None, "reused": False, "scope_verified": False}
    differences = pd.DataFrame(columns=["county_fips", "scope_status"])
    if q1 is None:
        report["reason"] = "Q1 file was not provided; independently use official PLACES. Q1 scope/field convention comparison remains pending."
        return places, report, differences
    try:
        fips_col = col(q1, "county_fips", "CountyFIPS")
        candidate = q1.copy()
        candidate["county_fips"] = normalize_fips(q1[fips_col], "Q1")
        qa, pa = set(candidate.county_fips), set(places.county_fips)
        differences = pd.DataFrame([
            *({"county_fips": f, "scope_status": "q1_only"} for f in sorted(qa - pa)),
            *({"county_fips": f, "scope_status": "official_places_only"} for f in sorted(pa - qa)),
        ], columns=["county_fips", "scope_status"])
        report.update({"q1_counties": len(qa), "official_places_counties": len(pa),
                       "q1_only_count": len(qa - pa), "official_places_only_count": len(pa - qa),
                       "scope_verified": qa == pa})
        missing = [c for c in HEALTH_COLS if c not in candidate]
        report["missing_canonical_fields"] = missing
        if missing or qa != pa:
            report["reason"] = "Q1 fields/scope differ. Use official PLACES; all scope differences are exported. Team reconciliation is required before claiming Q1 alignment."
            return places, report, differences
        candidate = candidate.set_index("county_fips").sort_index()
        official = places.set_index("county_fips").sort_index()
        for c in HEALTH_COLS:
            candidate[c] = pd.to_numeric(candidate[c], errors="coerce").astype("Float64")
        pd.testing.assert_frame_equal(candidate[HEALTH_COLS], official[HEALTH_COLS],
                                      check_dtype=False, check_exact=False, rtol=0, atol=1e-9)
        official.loc[:, HEALTH_COLS] = candidate[HEALTH_COLS]
        report.update({"reused": True, "reason": "Q1 point estimates and CIs match the official release and full geographic scope; verified values reused; source file untouched."})
        return official.reset_index(), report, differences
    except (ValueError, AssertionError) as exc:
        report["reason"] = f"Q1 cannot be safely reused: {exc}"
        return places, report, differences


def merge_counties(cleaned, q1=None):
    places, q1_report, scope_diff = check_q1(q1, cleaned["places"])
    merged = places.merge(cleaned["svi"], on="county_fips", how="outer", validate="one_to_one", indicator=True)
    merged["join_status"] = merged.pop("_merge").astype("string").replace({"both": "matched", "left_only": "places_only", "right_only": "svi_only"})
    merged["state"] = merged.state.combine_first(merged.svi_state)
    merged["county"] = merged.county.combine_first(merged.svi_county)
    merged["is_nyc"] = merged.county_fips.isin(NYC)
    merged["borough"] = merged.county_fips.map(NYC).astype("string")
    state_conflicts = merged.loc[merged.svi_state.notna() & merged.state.notna() & merged.svi_state.ne(merged.state),
                                 ["county_fips", "state", "svi_state"]]
    if not state_conflicts.empty:
        raise ValueError(f"State abbreviations conflict across sources: {state_conflicts.to_dict('records')}")
    merged = merged.drop(columns=["svi_state", "svi_county"]).sort_values("county_fips").reset_index(drop=True)
    return merged, q1_report, scope_diff


In [8]:
merged, q1_report, scope_diff = merge_counties(cleaned, q1)
display(merged.head(12))
display(merged.join_status.value_counts(dropna=False).rename_axis("join_status").reset_index(name="n_counties"))
display(merged.loc[merged.join_status.ne("matched"), ["county_fips", "state", "county", "join_status"]])
display(pd.DataFrame([q1_report]))
display(scope_diff)
print("Merged counties:", len(merged))


Merged counties: 3144


,county_fips,state,county,pop18plus,DIABETES_adj,DIABETES_adj_lo,DIABETES_adj_hi,CHD_adj,CHD_adj_lo,CHD_adj_hi,BPHIGH_adj,BPHIGH_adj_lo,BPHIGH_adj_hi,OBESITY_adj,OBESITY_adj_lo,OBESITY_adj_hi,COPD_adj,COPD_adj_lo,COPD_adj_hi,RPL_THEMES,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,join_status,is_nyc,borough
0,01001,AL,Autauga,46253.0,11.4,9.8,13.2,5.6,4.9,6.4,38.2,33.9,42.4,39.5,32.0,47.2,6.4,5.4,7.6,0.2663,0.2367,0.3675,0.6354,0.2358,matched,False,<NA>
1,01003,AL,Baldwin,200397.0,10.4,8.9,11.9,5.5,4.8,6.3,37.0,33.3,40.5,35.0,29.6,40.5,6.0,4.9,7.2,0.3487,0.3433,0.2883,0.489,0.4308,matched,False,<NA>
2,01005,AL,Barbour,19517.0,16.5,14.5,18.8,7.4,6.7,8.3,45.4,40.9,49.9,47.1,38.8,55.4,9.3,8.1,10.7,0.9927,0.9399,0.9399,0.8941,0.9959,matched,False,<NA>
3,01007,AL,Bibb,17522.0,13.9,12.0,16.1,7.0,6.2,8.0,42.4,37.7,46.9,43.7,35.2,52.4,8.9,7.6,10.3,0.8451,0.8893,0.4954,0.6185,0.8931,matched,False,<NA>
4,01009,AL,Blount,46099.0,11.3,9.7,13.2,6.3,5.5,7.2,36.0,31.8,40.4,36.0,28.2,44.3,7.6,6.3,9.1,0.6166,0.6866,0.7884,0.413,0.3532,matched,False,<NA>
5,01011,AL,Bullock,7878.0,21.3,18.9,23.9,8.8,7.9,9.8,52.4,47.8,57.0,51.0,42.2,60.1,11.2,9.8,12.8,0.8985,0.8654,0.5138,0.9809,0.9491,matched,False,<NA>
6,01013,AL,Butler,14276.0,15.2,13.2,17.3,6.7,6.0,7.5,44.8,40.3,49.2,40.7,32.5,48.9,8.3,7.2,9.5,0.6872,0.8441,0.6106,0.8661,0.217,matched,False,<NA>
7,01015,AL,Calhoun,91555.0,12.9,11.2,14.9,6.4,5.7,7.2,41.5,37.3,45.8,39.1,32.3,46.4,7.8,6.7,9.1,0.8145,0.854,0.65,0.658,0.7114,matched,False,<NA>
8,01017,AL,Chambers,26970.0,14.5,12.5,16.5,6.5,5.8,7.3,42.8,38.4,47.4,41.6,33.5,50.0,8.0,6.9,9.3,0.8295,0.7171,0.5317,0.8396,0.9469,matched,False,<NA>
9,01019,AL,Cherokee,20877.0,11.8,10.1,13.7,6.6,5.8,7.6,37.5,33.2,42.0,39.3,31.1,47.9,8.2,6.8,9.7,0.406,0.6077,0.6494,0.2628,0.1152,matched,False,<NA>


,join_status,n_counties
0,matched,3143
1,svi_only,1


,county_fips,state,county,join_status
2675,48301,TX,Loving County,svi_only


,exists,reused,scope_verified,reason
0,False,False,False,Q1 file was not provided; independently use of...


,county_fips,scope_status


## 4. Q3 Transformation

The shared table retains all five disease estimates, their CIs, adult population, overall and theme SVI ranks, and NYC flags. Borough names follow Q2; the original county names are retained separately. The fixed 12-row sample contains all five NYC counties and seven other named counties, selected for contextual breadth rather than statistical representativeness.

Compute five overall-SVI Spearman correlations using pairwise complete observations. Rank **after** selecting the complete pair and use average ranks for ties. Each county has equal weight; NYC contributes to the national pairs when complete. No population weighting, imputation or causal interpretation.


In [23]:
def correlations(df, predictors):
    """Spearman rho = Pearson correlation of within-pair average ranks.

    Rank after pairwise deletion; otherwise missingness changes ranks incorrectly.
    No weights, imputation, p-values, or causal interpretations.
    """
    records = []
    for theme in predictors:
        for disease in DISEASES:
            pair = df[[theme, f"{disease}_adj"]].dropna()
            n = len(pair)
            rho = None
            if n >= 3 and pair.nunique().min() > 1:
                value = pair.rank(method="average").astype(float).corr(method="pearson").iloc[0, 1]
                rho = float(value) if math.isfinite(value) else None
            records.append({"theme": theme, "disease": disease, "correlation": rho, "n_counties": n})
    return pd.DataFrame(records)


def q3_transform(merged):
    # Retain all 4 theme ranks here so Q4 uses exactly the same merged snapshot.
    q3 = merged[["county_fips", "state", "county", "pop18plus", *HEALTH_COLS,
                 *RANKS, "is_nyc", "borough", "join_status"]].copy()
    summary = correlations(q3, ["RPL_THEMES"]).rename(columns={"theme": "svi_measure"})
    sample = q3.set_index("county_fips").reindex(SAMPLE_FIPS)
    missing_sample = sample["county"].isna()
    if missing_sample.any():
        raise ValueError(f"Fixed documentation sample counties absent or unnamed: {sample.index[missing_sample].tolist()}")
    return q3, summary, sample.reset_index()


In [10]:
q3, summary, sample = q3_transform(merged)
display(q3.head(12))
display(sample)  # full real sample, including all ten CI bounds
display(summary)
display(q3.loc[q3.is_nyc, ["county_fips", "county", "borough", "RPL_THEMES", *VALUE_COLS, "join_status"]])


,county_fips,state,county,pop18plus,DIABETES_adj,DIABETES_adj_lo,DIABETES_adj_hi,CHD_adj,CHD_adj_lo,CHD_adj_hi,BPHIGH_adj,BPHIGH_adj_lo,BPHIGH_adj_hi,OBESITY_adj,OBESITY_adj_lo,OBESITY_adj_hi,COPD_adj,COPD_adj_lo,COPD_adj_hi,RPL_THEMES,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,is_nyc,borough,join_status
0,01001,AL,Autauga,46253.0,11.4,9.8,13.2,5.6,4.9,6.4,38.2,33.9,42.4,39.5,32.0,47.2,6.4,5.4,7.6,0.2663,0.2367,0.3675,0.6354,0.2358,False,<NA>,matched
1,01003,AL,Baldwin,200397.0,10.4,8.9,11.9,5.5,4.8,6.3,37.0,33.3,40.5,35.0,29.6,40.5,6.0,4.9,7.2,0.3487,0.3433,0.2883,0.489,0.4308,False,<NA>,matched
2,01005,AL,Barbour,19517.0,16.5,14.5,18.8,7.4,6.7,8.3,45.4,40.9,49.9,47.1,38.8,55.4,9.3,8.1,10.7,0.9927,0.9399,0.9399,0.8941,0.9959,False,<NA>,matched
3,01007,AL,Bibb,17522.0,13.9,12.0,16.1,7.0,6.2,8.0,42.4,37.7,46.9,43.7,35.2,52.4,8.9,7.6,10.3,0.8451,0.8893,0.4954,0.6185,0.8931,False,<NA>,matched
4,01009,AL,Blount,46099.0,11.3,9.7,13.2,6.3,5.5,7.2,36.0,31.8,40.4,36.0,28.2,44.3,7.6,6.3,9.1,0.6166,0.6866,0.7884,0.413,0.3532,False,<NA>,matched
5,01011,AL,Bullock,7878.0,21.3,18.9,23.9,8.8,7.9,9.8,52.4,47.8,57.0,51.0,42.2,60.1,11.2,9.8,12.8,0.8985,0.8654,0.5138,0.9809,0.9491,False,<NA>,matched
6,01013,AL,Butler,14276.0,15.2,13.2,17.3,6.7,6.0,7.5,44.8,40.3,49.2,40.7,32.5,48.9,8.3,7.2,9.5,0.6872,0.8441,0.6106,0.8661,0.217,False,<NA>,matched
7,01015,AL,Calhoun,91555.0,12.9,11.2,14.9,6.4,5.7,7.2,41.5,37.3,45.8,39.1,32.3,46.4,7.8,6.7,9.1,0.8145,0.854,0.65,0.658,0.7114,False,<NA>,matched
8,01017,AL,Chambers,26970.0,14.5,12.5,16.5,6.5,5.8,7.3,42.8,38.4,47.4,41.6,33.5,50.0,8.0,6.9,9.3,0.8295,0.7171,0.5317,0.8396,0.9469,False,<NA>,matched
9,01019,AL,Cherokee,20877.0,11.8,10.1,13.7,6.6,5.8,7.6,37.5,33.2,42.0,39.3,31.1,47.9,8.2,6.8,9.7,0.406,0.6077,0.6494,0.2628,0.1152,False,<NA>,matched


,county_fips,state,county,pop18plus,DIABETES_adj,DIABETES_adj_lo,DIABETES_adj_hi,CHD_adj,CHD_adj_lo,CHD_adj_hi,BPHIGH_adj,BPHIGH_adj_lo,BPHIGH_adj_hi,OBESITY_adj,OBESITY_adj_lo,OBESITY_adj_hi,COPD_adj,COPD_adj_lo,COPD_adj_hi,RPL_THEMES,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,is_nyc,borough,join_status
0,36005,NY,Bronx,1031703.0,14.8,13.3,16.6,6.3,5.6,7.1,34.6,32.0,37.2,33.7,30.0,37.4,6.4,5.5,7.3,0.9965,0.9787,0.9545,0.9952,0.9806,True,Bronx,matched
1,36047,NY,Kings,2003937.0,11.1,9.8,12.5,5.1,4.5,5.7,28.8,26.6,31.1,25.0,22.2,28.0,5.4,4.6,6.3,0.8874,0.8775,0.5352,0.9395,0.9249,True,Brooklyn,matched
2,36061,NY,New York,1381420.0,8.4,7.5,9.4,4.4,3.9,4.9,25.8,23.8,27.9,19.5,17.1,22.0,4.1,3.4,4.8,0.7283,0.6268,0.0996,0.8896,0.9784,True,Manhattan,matched
3,36081,NY,Queens,1821507.0,12.2,10.8,13.6,4.9,4.3,5.4,29.8,27.5,32.1,23.8,21.1,26.6,4.6,4.0,5.3,0.8024,0.8282,0.1899,0.9765,0.8772,True,Queens,matched
4,36085,NY,Richmond,385428.0,10.1,8.8,11.5,4.8,4.2,5.5,30.3,27.6,33.1,30.0,26.2,34.1,5.2,4.3,6.0,0.5956,0.4413,0.3328,0.8021,0.811,True,Staten Island,matched
5,06037,CA,Los Angeles,7710575.0,11.6,10.4,12.8,4.8,4.3,5.4,28.4,26.4,30.5,26.4,23.7,29.4,4.5,3.9,5.2,0.8629,0.895,0.2825,0.9736,0.9176,False,<NA>,matched
6,17031,IL,Cook,4031580.0,10.7,9.5,12.0,4.7,4.2,5.3,28.9,26.5,31.3,31.0,27.6,34.5,5.2,4.4,6.0,0.7964,0.7754,0.3926,0.9138,0.8298,False,<NA>,matched
7,48201,TX,Harris,3595915.0,13.2,11.6,14.9,5.5,4.9,6.2,32.9,29.9,36.0,37.1,32.6,41.6,5.5,4.7,6.4,0.9201,0.965,0.7458,0.9672,0.7089,False,<NA>,matched
8,04013,AZ,Maricopa,3572375.0,9.6,8.6,10.7,5.0,4.4,5.6,28.6,26.5,30.8,30.6,27.2,34.2,5.2,4.4,6.1,0.713,0.6201,0.5705,0.8521,0.6907,False,<NA>,matched
9,12086,FL,Miami-Dade,2154272.0,13.3,11.6,15.2,5.7,5.0,6.4,30.3,27.2,33.6,29.3,24.6,34.2,4.7,3.9,5.5,0.881,0.9049,0.4365,0.9901,0.8909,False,<NA>,matched


,svi_measure,disease,correlation,n_counties
0,RPL_THEMES,DIABETES,0.751382,2956
1,RPL_THEMES,CHD,0.618833,2956
2,RPL_THEMES,BPHIGH,0.600008,2956
3,RPL_THEMES,OBESITY,0.340842,2956
4,RPL_THEMES,COPD,0.496396,2956


,county_fips,county,borough,RPL_THEMES,DIABETES_adj,CHD_adj,BPHIGH_adj,OBESITY_adj,COPD_adj,join_status
1832,36005,Bronx,Bronx,0.9965,14.8,6.3,34.6,33.7,6.4,matched
1853,36047,Kings,Brooklyn,0.8874,11.1,5.1,28.8,25.0,5.4,matched
1860,36061,New York,Manhattan,0.7283,8.4,4.4,25.8,19.5,4.1,matched
1870,36081,Queens,Queens,0.8024,12.2,4.9,29.8,23.8,4.6,matched
1872,36085,Richmond,Staten Island,0.5956,10.1,4.8,30.3,30.0,5.2,matched


## 5. Q4 Correlation Analysis

Reuse Q3 directly. Four themes × five conditions yield 20 coefficients. Spearman rho is the Pearson correlation of within-pair average ranks. Pair-specific n accompanies each coefficient. Undefined correlations (fewer than three pairs or constant values) remain missing, not zero. These descriptive associations do not prove causation or significance; theme differences may reflect different complete-case samples.

Theme 1: Socioeconomic Status; Theme 2: Household Characteristics; Theme 3: Racial and Ethnic Minority Status; Theme 4: Housing Type and Transportation.


In [24]:
def q4_transform(q3):
    tidy = correlations(q3, THEMES)
    matrix = tidy.pivot(index="theme", columns="disease", values="correlation").reindex(index=THEMES, columns=DISEASES)
    counts = tidy.pivot(index="theme", columns="disease", values="n_counties").reindex(index=THEMES, columns=DISEASES)
    return matrix, tidy, counts


In [12]:
matrix, tidy, counts = q4_transform(q3)
display(matrix)   # complete 4 x 5 coefficient matrix, no chart
display(counts)   # corresponding effective n matrix
display(tidy)     # all 20 theme / disease / correlation / n records


disease,DIABETES,CHD,BPHIGH,OBESITY,COPD
theme,,,,,
RPL_THEME1,0.785865,0.708812,0.664094,0.375265,0.615456
RPL_THEME2,0.555494,0.476414,0.442146,0.332060,0.385154
RPL_THEME3,0.508056,0.162590,0.309988,0.069114,-0.016403
RPL_THEME4,0.427757,0.354522,0.324102,0.167909,0.268512


disease,DIABETES,CHD,BPHIGH,OBESITY,COPD
theme,,,,,
RPL_THEME1,2956,2956,2956,2956,2956
RPL_THEME2,2956,2956,2956,2956,2956
RPL_THEME3,2956,2956,2956,2956,2956
RPL_THEME4,2956,2956,2956,2956,2956


,theme,disease,correlation,n_counties
0,RPL_THEME1,DIABETES,0.785865,2956
1,RPL_THEME1,CHD,0.708812,2956
2,RPL_THEME1,BPHIGH,0.664094,2956
3,RPL_THEME1,OBESITY,0.375265,2956
4,RPL_THEME1,COPD,0.615456,2956
5,RPL_THEME2,DIABETES,0.555494,2956
6,RPL_THEME2,CHD,0.476414,2956
7,RPL_THEME2,BPHIGH,0.442146,2956
8,RPL_THEME2,OBESITY,0.332060,2956
9,RPL_THEME2,COPD,0.385154,2956


## 6. Validation

Check raw/merged county counts, FIPS match rates with explicit denominators, duplicate FIPS, retention of in-scope counties, NYC five-county presence, valid ranges, CI ordering, missingness and all 25 pairwise n. Matrix and tidy values must agree. Out-of-range values are made missing with original values in the audit. Structural FIPS checks do not independently validate every code against a census boundary vintage.

**Interpretation limits:** PLACES 2025 is a release label; these five conditions use 2023 BRFSS-based estimates, whereas SVI 2022 uses ACS 2018–2022. PLACES outcomes are model-based. PLACES inputs (age, race/ethnicity, education and county poverty among others) overlap with SVI themes. County ecological associations, spatial dependence and unpropagated model-estimation uncertainty limit interpretation. Missing estimates can restrict national representativeness. This stage performs no spatial modeling, p-value testing or causal inference.

Startup's reported 3,143 PLACES counties / 187 missing estimates are comparison context; the code calculates actual counts instead of forcing agreement. In the supplied files, SVI includes Loving County TX (`48301`), absent from PLACES. The outer join keeps it as `svi_only`; 187 KY/PA counties plus Loving therefore have missing disease estimates in the shared 3,144-row table. Inspect the actual tables below when rerunning with a different snapshot.


In [25]:
def validate(frames, cleaned, merged, q3, summary, matrix, tidy, issues, exclusions, q1_report):
    checks = []
    def check(name, passed, detail):
        checks.append({"check": name, "passed": bool(passed), "detail": detail})
    check("unique_merged_fips", not merged.county_fips.duplicated().any(), len(merged))
    check("five_digit_fips", merged.county_fips.str.fullmatch(r"\d{5}").all(), "5-digit strings")
    check("national_states_only", merged.county_fips.str[:2].isin(US_STATES).all(), "50 states + DC")
    check("all_source_counties_retained", set(merged.county_fips) == set(cleaned["places"].county_fips) | set(cleaned["svi"].county_fips), "Outer join; no county removed for missing values")
    check("nyc_five_present", set(NYC).issubset(set(merged.county_fips)), sorted(NYC))
    nyc = merged[merged.is_nyc]
    check("nyc_all_sources_matched", len(nyc) == 5 and nyc.join_status.eq("matched").all(), "Five NYC FIPS must occur in both sources")
    check("nyc_boroughs_only", q3.loc[q3.is_nyc, "borough"].notna().all() and q3.loc[~q3.is_nyc, "borough"].isna().all(), "Official county names retained; borough labels added separately")
    population = q3.pop18plus.dropna()
    check("adult_population_valid", (population.ge(0) & population.mod(1).eq(0)).all(), "Nullable nonnegative integer; context only")
    for c in HEALTH_COLS + RANKS:
        hi = 1 if c in RANKS else 100
        check(f"range_{c}", q3[c].dropna().between(0, hi).all(), f"Nonmissing range [0, {hi}]")
    for d in DISEASES:
        a, lo, hi = (q3[f"{d}_adj{s}"] for s in ("", "_lo", "_hi"))
        valid = ((lo.isna() | hi.isna() | lo.le(hi)) & (a.isna() | lo.isna() | a.ge(lo)) & (a.isna() | hi.isna() | a.le(hi)))
        check(f"ci_order_{d}", valid.fillna(False).all(), "lower <= estimate <= upper when available")
    check("q4_shape", matrix.shape == (4, 5) and len(tidy) == 20, list(matrix.shape))
    check("q4_unique_pairs", not tidy.duplicated(["theme", "disease"]).any(), "20 distinct theme-disease pairs")
    check("q4_matrix_tidy_agree", all(
        (pd.isna(row.correlation) and pd.isna(matrix.loc[row.theme, row.disease])) or
        (not pd.isna(row.correlation) and row.correlation == matrix.loc[row.theme, row.disease])
        for row in tidy.itertuples()), "All 20 matrix cells agree with tidy rows")
    check("q3_five_correlations", len(summary) == 5, len(summary))
    combined = pd.concat([summary.rename(columns={"svi_measure": "theme"}), tidy], ignore_index=True)
    for row in combined.itertuples():
        expected = int(q3[[row.theme, f"{row.disease}_adj"]].notna().all(axis=1).sum())
        check(f"pair_n_{row.theme}_{row.disease}", row.n_counties == expected, expected)
    check("rho_range", combined.correlation.dropna().between(-1, 1).all(), "[-1, 1]")
    matched = int(merged.join_status.eq("matched").sum())
    info = {
        "run_utc": utc_now(), "status": "completed" if all(c["passed"] for c in checks) else "validation_failed",
        "raw": {k: {"rows": len(v), "duplicate_full_rows": int(v.duplicated().sum())} for k, v in frames.items()},
        "in_scope": {k: len(v) for k, v in cleaned.items()},
        "merged_counties": len(merged), "matched_counties": matched,
        "places_only_counties": int(merged.join_status.eq("places_only").sum()),
        "svi_only_counties": int(merged.join_status.eq("svi_only").sum()),
        "fips_match_rate_places_denominator": matched / len(cleaned["places"]),
        "fips_match_rate_svi_denominator": matched / len(cleaned["svi"]),
        "fips_match_rate_union_denominator": matched / len(merged),
        "duplicate_fips_after_cleaning": {k: int(v.county_fips.duplicated().sum()) for k, v in cleaned.items()},
        "duplicate_fips_merged": int(merged.county_fips.duplicated().sum()),
        "missing_counts": {c: int(n) for c, n in q3.isna().sum().items()},
        "issue_counts": {str(k): int(v) for k, v in issues.reason.value_counts().items()},
        "explicit_scope_exclusions": len(exclusions), "q1": q1_report,
        "observed_ranges": {c: {"min": None if q3[c].dropna().empty else float(q3[c].min()),
                                 "max": None if q3[c].dropna().empty else float(q3[c].max())} for c in HEALTH_COLS + RANKS},
        "nyc": {f: {"present": bool(merged.county_fips.eq(f).any()), "borough": b} for f, b in NYC.items()},
        "correlations": combined.astype(object).where(combined.notna(), None).to_dict("records"),
        "checks": checks,
    }
    if not all(c["passed"] for c in checks):
        failed = [c["check"] for c in checks if not c["passed"]]
        raise ValueError(f"Validation failed; processed CSVs not exported: {failed}")
    return info, pd.DataFrame(checks)


In [14]:
audit, checks = validate(frames, cleaned, merged, q3, summary, matrix, tidy, issues, exclusions, q1_report)
display(checks)
display(pd.DataFrame([{"metric": k, "value": audit[k]} for k in [
    "merged_counties", "matched_counties", "places_only_counties", "svi_only_counties",
    "fips_match_rate_places_denominator", "fips_match_rate_svi_denominator",
    "fips_match_rate_union_denominator", "duplicate_fips_merged"]]))
display(q3.isna().sum().rename("n_missing").rename_axis("column").reset_index())
display(pd.DataFrame(audit["observed_ranges"]).T)
display(pd.DataFrame(audit["correlations"]))
display(issues.loc[issues.reason.isin(["numeric_parse_error", "out_of_range_or_noninteger", "malformed_ci", "ci_order_or_estimate_outside_ci"])])
display(pd.DataFrame([{"state": state, "n_counties": len(g), **{c: int(g[c].isna().sum()) for c in VALUE_COLS}} for state, g in q3.groupby("state", dropna=False)]))


,check,passed,detail
0,unique_merged_fips,True,3144
1,five_digit_fips,True,5-digit strings
2,national_states_only,True,50 states + DC
3,all_source_counties_retained,True,Outer join; no county removed for missing values
4,nyc_five_present,True,"[36005, 36047, 36061, 36081, 36085]"
5,nyc_all_sources_matched,True,Five NYC FIPS must occur in both sources
6,nyc_boroughs_only,True,Official county names retained; borough labels...
7,adult_population_valid,True,Nullable nonnegative integer; context only
8,range_DIABETES_adj,True,"Nonmissing range [0, 100]"
9,range_DIABETES_adj_lo,True,"Nonmissing range [0, 100]"


,metric,value
0,merged_counties,3144.000000
1,matched_counties,3143.000000
2,places_only_counties,0.000000
3,svi_only_counties,1.000000
4,fips_match_rate_places_denominator,1.000000
5,fips_match_rate_svi_denominator,0.999682
6,fips_match_rate_union_denominator,0.999682
7,duplicate_fips_merged,0.000000


,column,n_missing
0,county_fips,0
1,state,0
2,county,0
3,pop18plus,1
4,DIABETES_adj,188
5,DIABETES_adj_lo,188
6,DIABETES_adj_hi,188
7,CHD_adj,188
8,CHD_adj_lo,188
9,CHD_adj_hi,188


,min,max
DIABETES_adj,6.0,23.8
DIABETES_adj_lo,5.1,21.3
DIABETES_adj_hi,7.1,26.6
CHD_adj,3.8,11.4
CHD_adj_lo,3.4,10.2
CHD_adj_hi,4.4,12.8
BPHIGH_adj,21.0,53.1
BPHIGH_adj_lo,18.6,48.3
BPHIGH_adj_hi,23.6,57.6
OBESITY_adj,17.1,54.0


,theme,disease,correlation,n_counties
0,RPL_THEMES,DIABETES,0.751382,2956
1,RPL_THEMES,CHD,0.618833,2956
2,RPL_THEMES,BPHIGH,0.600008,2956
3,RPL_THEMES,OBESITY,0.340842,2956
4,RPL_THEMES,COPD,0.496396,2956
5,RPL_THEME1,DIABETES,0.785865,2956
6,RPL_THEME1,CHD,0.708812,2956
7,RPL_THEME1,BPHIGH,0.664094,2956
8,RPL_THEME1,OBESITY,0.375265,2956
9,RPL_THEME1,COPD,0.615456,2956


,dataset,county_fips,field,raw_value,reason


,state,n_counties,DIABETES_adj,CHD_adj,BPHIGH_adj,OBESITY_adj,COPD_adj
0,AK,30,0,0,0,0,0
1,AL,67,0,0,0,0,0
2,AR,75,0,0,0,0,0
3,AZ,15,0,0,0,0,0
4,CA,58,0,0,0,0,0
5,CO,64,0,0,0,0,0
6,CT,9,0,0,0,0,0
7,DC,1,0,0,0,0,0
8,DE,3,0,0,0,0,0
9,FL,67,0,0,0,0,0


## 7. Export

Write only the four final UTF-8 CSVs to data/processed, without an index. Sample rows are selected from q3_county_svi.csv; pairwise n is included in q4_correlations_tidy.csv. Machine-readable provenance/audits go to docs/validation. Blank CSV fields mean missing; never convert them to zero in a visualization. The notes contain Plan / Data Table / Table Details and future view/interaction descriptions only. Actual tables are inserted only after successful validation.

This notebook carries the documentation template and standalone Python script so it can be run without a clone. Existing notes are preserved apart from the marked generated-results block. No Q1 file is overwritten. The final optional Colab ZIP contains outputs, notes and script; use **File → Download → .ipynb** separately to retain this notebook's executed outputs.

**AI disclosure:** AI assisted in preparing the transformation code, notebook and notes from the provided materials. Empirical values must come from official CDC files. No synthetic data or fabricated results are used; human review is required. The exported notes record this statement for the final README.


In [26]:
def markdown_table(df, digits=4):
    # Avoid an optional tabulate dependency in Colab.
    def fmt(v):
        if pd.isna(v):
            return "NA"
        if isinstance(v, float):
            return f"{v:.{digits}f}"
        return str(v).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "| " + " | ".join(["---"] * len(df.columns)) + " |"]
    lines.extend("| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False, name=None))
    return "\n".join(lines)


def write_results_section(root, sample, summary, matrix, tidy, counts, audit):
    notes = root / "q3_q4_transformation_notes.md"
    if not notes.exists():
        raise FileNotFoundError("Keep q3_q4_transformation_notes.md with the repository; it contains definitions and limitations.")
    marker = "<!-- GENERATED_RESULTS_START -->"
    endmarker = "<!-- GENERATED_RESULTS_END -->"
    text = notes.read_text(encoding="utf-8")
    if marker not in text or endmarker not in text:
        raise ValueError("Notes results markers missing")
    section = (
        f"\n\nActual run completed at {audit['run_utc']}. These tables are generated from the cached raw CSVs.\n\n"
        "#### Q3 Data Table — fixed 12-county documentation sample\n\n"
        "NYC five counties plus seven named counties chosen for geographical/contextual breadth, not statistical representativeness. "
        "These rows are selected directly from data/processed/q3_county_svi.csv, which also contains adult population, all CIs, four themes and join status.\n\n"
        + markdown_table(sample[["county_fips", "state", "county", *VALUE_COLS, "RPL_THEMES", "is_nyc", "borough"]])
        + "\n\nQ3 overall SVI Spearman correlations:\n\n" + markdown_table(summary)
        + "\n\n#### Q4 Data Table — Spearman correlation matrix\n\n" + markdown_table(matrix.reset_index())
        + "\n\nPairwise valid county counts:\n\n" + markdown_table(counts.reset_index())
        + "\n\nQ4 complete tidy table:\n\n" + markdown_table(tidy)
        + "\n\n#### Actual validation\n\n"
        + markdown_table(pd.DataFrame([
            {"metric": key, "value": audit[key]} for key in ["merged_counties", "matched_counties", "places_only_counties", "svi_only_counties",
             "fips_match_rate_places_denominator", "fips_match_rate_svi_denominator", "duplicate_fips_merged", "explicit_scope_exclusions"]]))
        + "\n\nRaw source counts:\n\n" + markdown_table(pd.DataFrame([{"dataset": k, **v} for k, v in audit["raw"].items()]))
        + "\n\nMissing values (non-NYC borough blanks are expected):\n\n"
        + markdown_table(pd.DataFrame([{"column": k, "n_missing": v} for k, v in audit["missing_counts"].items()]))
        + "\n\nQ1 alignment/reuse status: `" + json.dumps(audit["q1"], ensure_ascii=False).replace("`", "'") + "`\n\n"
        + "Full checks, anomaly records, out-of-scope records, unmatched FIPS and source hashes are in the exported audit files.\n\n"
    )
    before, rest = text.split(marker, 1)
    _, after = rest.split(endmarker, 1)
    result = before + marker + section + endmarker + after
    status_start, status_end = "<!-- RUN_STATUS_START -->", "<!-- RUN_STATUS_END -->"
    if status_start in result and status_end in result:
        before_status, rest_status = result.split(status_start, 1)
        _, after_status = rest_status.split(status_end, 1)
        result = (before_status + status_start + f"\n\n**Current status: empirical transformation completed at {audit['run_utc']}; implemented validation checks passed.**\n\n"
                  "Actual computed results are in the generated-results section. Source hashes and download records are in the run manifest. "
                  "An uploaded file's release year and national ranking universe still require confirmation against the original CDC export. "
                  "Q1 alignment status is recorded separately and may remain unresolved.\n\n" + status_end + after_status)
    notes.write_text(result, encoding="utf-8")


def export_results(root, q3, summary, sample, matrix, tidy, counts, audit, checks,
                   issues, exclusions, scope_diff, inspection, raw_summary, provenance):
    root = Path(root)
    out = root / "data/processed"
    out.mkdir(parents=True, exist_ok=True)
    validation_dir = root / "docs/validation"
    validation_dir.mkdir(parents=True, exist_ok=True)
    outputs = {
        "q3_county_svi.csv": q3, "q3_national_summary.csv": summary,
        "q4_correlations.csv": matrix.reset_index(), "q4_correlations_tidy.csv": tidy,
    }
    diagnostics = {
        "unmatched_counties.csv": q3.loc[q3.join_status.ne("matched"), ["county_fips", "state", "county", "join_status"]],
        "missingness.csv": pd.DataFrame([{"column": c, "n_missing": n, "n_counties": len(q3)} for c, n in audit["missing_counts"].items()]),
        "cleaning_issues.csv": issues, "scope_exclusions.csv": exclusions,
        "q1_scope_differences.csv": scope_diff, "raw_schema.csv": inspection,
        "raw_summary.csv": raw_summary, "validation_checks.csv": checks,
    }
    # Write notes first to catch missing documentation templates before CSV export.
    write_results_section(root, sample, summary, matrix, tidy, counts, audit)
    for filename, df in outputs.items():
        df.to_csv(out / filename, index=False, encoding="utf-8", na_rep="", float_format="%.10g")
    for filename, df in diagnostics.items():
        df.to_csv(validation_dir / filename, index=False, encoding="utf-8", na_rep="", float_format="%.10g")
    json_write(validation_dir / "validation.json", audit)
    manifest = {"run_utc": audit["run_utc"], "status": audit["status"], "sources": provenance,
                "software": {"python": sys.version.split()[0], "pandas": pd.__version__},
                "parameters": {"scope": "Union of official county FIPS in 50 states + DC", "join": "outer one_to_one",
                               "correlation": "Spearman; pairwise complete; average ranks for ties", "weighted": False,
                               "places_release": 2025, "svi_release": 2022,
                               "nyc_borough_map": NYC, "sample_fips": SAMPLE_FIPS},
                "outputs": {name: {"rows": len(df), "sha256": file_sha256(out / name)} for name, df in outputs.items()},
                "validation_files": {name: {"rows": len(df), "sha256": file_sha256(validation_dir / name)} for name, df in diagnostics.items()}}
    json_write(validation_dir / "run_manifest.json", manifest)
    (validation_dir / "last_attempt.json").unlink(missing_ok=True)
    return list(outputs)


In [27]:
# Embedded reproducibility assets; this cell writes only this task's own files.
NOTES_TEMPLATE = '# BioVis · Milestone 2 · Q3/Q4 Data Transformation\n\n<!-- RUN_STATUS_START -->\n\n**Current status: empirical transformation completed at 2026-11-02T20:47:33.524736+00:00; implemented validation checks passed.**\n\nActual computed results are in the generated-results section. Source hashes and download records are in `docs/validation/run_manifest.json`. An uploaded file\'s release year and national ranking universe still require confirmation against the original CDC export. Q1 alignment status is recorded separately and may remain unresolved.\n\n<!-- RUN_STATUS_END -->\n\nThis document covers data transformation only. Views and interactions below are future plans. No sketches, visualizations, causal analysis, gap scores or final report are produced.\n\n## Reference alignment\n\nRead and used:\n\n- `Information Visualization Project Proposal.docx`: five outcomes, 2025 PLACES / 2022 SVI, Q3 and Q4 questions, national county comparisons and NYC highlighting.\n- `Team15.M1.feedback.pdf`, pp. 1–2: fix the analysis unit, identify shared model inputs, distinguish data periods, use association wording, document data access and AI use.\n- `Startup.pdf`, slides 12–13, 18–21, 25–28: Plan / Data Table / Table Details format; `county_fips`, `state`, `county`, `pop18plus`, `{m}_adj`, `{m}_adj_lo`, `{m}_adj_hi`; borough labels Bronx, Brooklyn, Manhattan, Queens, Staten Island.\n\nThe Q3/Q4 unit is **one US county or county equivalent**, including DC, Alaska and Hawaii; no Puerto Rico or other territories. NYC five counties are highlighted within the national county data. Q2/Q5/Q6 remain NYC census-tract analyses and are outside this work.\n\nThe proposal\'s older preprocessing paragraph refers broadly to an 11-digit tract join, removal of missing/small-population tracts and a gap score. Those steps do **not** apply here: this work follows the user\'s county-level specification and the feedback. SVI uses CDC\'s published national **county** rankings. Tract percentiles are never averaged. Missing counties/estimates are retained. The gap score stays outside the core, consistent with Startup slide 9.\n\nThe supplied PLACES CSV independently confirms Startup\'s reported 3,143 counties and 187 counties without each of the five selected estimates (120 KY, 67 PA). These counts were calculated from the actual CSV and were not hard-coded as cleaning targets. The supplied SVI CSV has 3,144 counties, including Loving County TX (`48301`), which is absent from PLACES. The shared outer join preserves Loving as `svi_only`, yielding 3,144 rows and 188 missing values per disease/CI column. A county-equivalent count must be read from the particular release, not inferred from a general current CDC page.\n\nStartup\'s source overview lists tract-level SVI and slide 5 allows an owner decision on county SVI. For Q3/Q4, this is resolved in favor of the official national county CSV, as the user requires. The slides should eventually identify this additional county-level SVI source.\n\n## Q3\n\n**Question.** How is chronic disease prevalence associated with overall social vulnerability across US counties, and how do NYC\'s five counties compare with the national pattern?\n\n### Plan\n\n**Unit and Data.** One county/county equivalent. CDC PLACES 2025 County GIS-Friendly (`i46a-9kgh`): age-adjusted adult prevalence and 95% CIs for diabetes, CHD, high blood pressure, obesity and COPD; adult population where provided. CDC/ATSDR SVI 2022: `RPL_THEMES`, ranked against the US county universe. County names remain official source labels; borough labels are separate.\n\n**Transformations.** Inspect actual headers, as-read types, rows and duplicates. Normalize valid county FIPS to five-character strings. Select published adjusted values and split adjusted CI strings. Convert source missing tokens and `-999` to NA; retain a cell-level audit. Convert other out-of-range/unparseable values to NA with their original values recorded. Outer-join once by FIPS, with one-to-one validation, and keep a join-status flag. Preserve all in-scope source counties and original missingness. Compute five national unweighted pairwise-complete Spearman coefficients. Add NYC and borough flags and a fixed 12-county documentation sample.\n\n**How it answers the question.** A common county table supports national association views with the same five NYC counties marked. The five coefficients summarize monotonic national patterns without claiming causation. NYC is included in each national calculation when its paired values are available; no separate correlation based on only five NYC counties is estimated.\n\n**Views — future plan only.** County scatterplot of `RPL_THEMES` (x: 0–1) against selected adjusted prevalence (y: %); distinguish NYC using labels and symbol outlines. A linked national county map may locate selected counties. Show missing-pair counts explicitly. CI fields support uncertainty inspection.\n\n**Interaction — future plan only.** Switch disease; hover to show county, adjusted estimate, 95% CI and SVI; highlight NYC; brush a scatterplot range to highlight corresponding counties on a linked map. Use fixed axes within a condition and label missing values. Q3/Q4 do not depend on a browser statistics engine.\n\n**Risks.** Different data periods; missing modeled estimates; partial overlap of PLACES inputs and SVI factors; geography vintage differences; ecological and spatial dependence; uncertainty not incorporated into the rank correlation; NYC is only five counties. No disease counts are estimated by multiplying age-adjusted prevalence by population.\n\n### Data Table\n\nThe actual 12-county sample and overall-SVI correlations are displayed in the generated-results section and exported as CSVs. The executed notebook also shows all sample columns, including the confidence limits.\n\nThe fixed sample selects NYC five counties plus Los Angeles CA (`06037`), Cook IL (`17031`), Harris TX (`48201`), Maricopa AZ (`04013`), Miami-Dade FL (`12086`), Hinds MS (`28049`) and Autauga AL (`01001`). It provides geographic/contextual breadth and is not a statistically representative sample. The selected rows in q3_county_svi.csv retain CIs and all shared fields, even when the compact Markdown table omits them for readability.\n\n### Table Details — Column Definitions\n\n| Column | Type / unit | Meaning / source |\n| --- | --- | --- |\n| `county_fips` | Text, five digits | State + county FIPS; join key; preserve leading zeros |\n| `state` | Text, postal abbreviation | PLACES `StateAbbr`; SVI `ST_ABBR` for SVI-only rows |\n| `county` | Text | PLACES `CountyName`; SVI `COUNTY` for SVI-only rows |\n| `pop18plus` | Count, nullable | PLACES `TotalPop18plus`, adults aged 18+, Census 2023 estimate according to CDC\'s release dictionary; context only, never a weight |\n| `DIABETES_adj` | Percent, 0–100 | Published age-adjusted adult diagnosed diabetes prevalence |\n| `CHD_adj` | Percent, 0–100 | Published age-adjusted adult coronary heart disease prevalence |\n| `BPHIGH_adj` | Percent, 0–100 | Published age-adjusted adult high blood pressure prevalence |\n| `OBESITY_adj` | Percent, 0–100 | Published age-adjusted adult obesity prevalence |\n| `COPD_adj` | Percent, 0–100 | Published age-adjusted adult COPD prevalence |\n| `{m}_adj_lo`, `{m}_adj_hi` | Percent, nullable | Published adjusted 95% confidence limits, parsed from `{m}_Adj95CI` |\n| `RPL_THEMES` | Number, 0–1 | CDC-published overall national county SVI percentile; higher = more vulnerable |\n| `RPL_THEME1` ... `RPL_THEME4` | Number, 0–1 | Four CDC-published national county theme percentiles, retained for Q4 reuse |\n| `is_nyc` | Boolean | FIPS in NYC five-county set |\n| `borough` | Text, nullable | `36005` Bronx; `36047` Brooklyn; `36061` Manhattan; `36081` Queens; `36085` Staten Island. Blank outside NYC is expected, not an error |\n| `join_status` | Text | `matched`, `places_only` or `svi_only`; measures join coverage, not completeness of disease/SVI values |\n\n`Kings` is the county name corresponding to Brooklyn; `New York` corresponds to Manhattan; `Richmond` corresponds to Staten Island. Keeping both avoids conflating borough and official county labels. No crude prevalence is substituted when an adjusted value is missing.\n\n`q3_national_summary.csv` contains `svi_measure`, `disease`, `correlation`, `n_counties`: one row per condition. In this question the summary is the requested overall-SVI association, rather than Q1\'s prevalence distribution statistics.\n\n### Table Details — Transformations\n\n1. Raw CSVs are retained without editing; SHA-256 hashes, actual headers and runtime versions are exported in the run manifest. Reads use pandas string dtype to protect FIPS and preserve original missing tokens. Numeric-parseability counts are reported separately from the intentionally string-based read dtype.\n2. Required columns are resolved case-insensitively so official display headers and API-style lowercase headers can be accepted. Unexpected schemas stop execution; the program never silently substitutes crude values or tract data. Numeric thousands separators are removed where present.\n3. FIPS are stripped, an integer-like `.0` suffix is removed, and valid 4/5-digit county codes are padded to length five. Missing, malformed, aggregate or duplicate FIPS stop the run; duplicates are not arbitrarily discarded. Both SVI `FIPS` and `STCNTY`, if present, must be unique and agree, which rejects tract files.\n4. Only 50 states + DC are in scope. Any other source rows are preserved in a separate exclusions table with a reason. Missing disease/SVI values never trigger removal. No small-population threshold is applied to counties.\n5. `-999`, blanks and documented textual missing markers become NA, never zero. Other numeric values outside allowed ranges are recorded as anomalies and masked to NA. Original source missing, sentinel, parse-error and invalid-value reasons remain distinguishable in the cleaning audit. Zeros within the valid range are retained.\n6. Parse CIs with a full-string expression. Malformed intervals, bounds outside 0–100, reversed intervals or intervals excluding their point estimate are audited. Inconsistent intervals are made missing while retaining the point estimate and county.\n7. Outer merge on FIPS with `validate=\'one_to_one\'` retains unmatched counties from both sources. State conflicts stop the run. The unweighted national analysis is based on each available pair in the merged county union. SVI-only rows cannot contribute without a PLACES estimate. No geometry download or boundary crosswalk is part of this stage.\n8. Q1 candidate inspection is performed before official downloads. Q1 values are reused only if all canonical adjusted values/CIs and the FIPS universe match the official release, including identical missingness. Otherwise use independent official PLACES processing, export Q1 scope differences and document unresolved alignment. The teammate file is never overwritten.\n\n### Table Details — Checks\n\nThe successful-run audit records raw and in-scope counts; full-row duplicates; duplicate county FIPS; matched and each unmatched count; match rates with explicit PLACES, SVI and union denominators; all five NYC counties; column missingness; anomalous source values; valid prevalence/SVI/CI ranges; Q1 field and scope compatibility; and effective n for all correlations. A matched FIPS does not guarantee nonmissing measures.\n\nThe outer-join universe can include SVI-only historical county codes. This may make Q3\'s retained rows differ from the Q1 PLACES universe. Such rows are explicitly flagged and listed; they are not assigned prevalence or mapped to new codes automatically. Connecticut planning-region versus historical-county FIPS, and any other county-equivalent changes, must be assessed from actual unmatched lists. Do not resolve them by averaging tract/county percentiles. If precise geography harmonization is needed, obtain compatible official county releases or a defensible separate crosswalk procedure before claiming complete national comparability.\n\n## Q4\n\n**Question.** Which of the four SVI themes show the strongest associations with chronic disease prevalence, and do these relationships differ across conditions?\n\n### Plan\n\n**Unit and Data.** The same county-level merged snapshot as Q3, with `RPL_THEME1` (Socioeconomic Status), `RPL_THEME2` (Household Characteristics), `RPL_THEME3` (Racial and Ethnic Minority Status), and `RPL_THEME4` (Housing Type and Transportation). No independently rebuilt merge or tract analysis.\n\n**Transformations.** For each of four themes × five conditions, select the two columns and remove only rows missing either member for that calculation. Rank both remaining variables with average ranks for ties, then compute Pearson correlation of the two rank vectors (Spearman rho). Keep n alongside each coefficient. Export the full 4 × 5 matrix, a matching n matrix and all 20 tidy rows. No population weighting, imputation or within-state re-ranking.\n\n**How it answers the question.** Columns allow comparison of theme association across conditions; rows allow comparison of conditions within a theme. Strength refers to absolute rho, while the sign distinguishes positive and negative associations. No strongest theme is named before real calculations. Different coefficients may use different county samples, so differences are descriptive and are not tests of coefficient differences.\n\n**Views — future plan only.** Labeled 4 × 5 correlation matrix with a fixed diverging scale from −1 to +1, numeric coefficients and n on inspection. A selected cell may open a county scatterplot. Missing/undefined values need an explicit label rather than a zero-colored cell.\n\n**Interaction — future plan only.** Select a theme-condition cell to display the corresponding scatterplot and valid n; hover to see theme definitions; optionally highlight NYC using the shared flag. An exploratory state filter would need separately precomputed summaries or an explicitly implemented future calculation; the current exported coefficients are national.\n\n**Risks.** Shared model inputs may contribute to association. Negative Theme 3 correlation would not indicate a protective causal effect of minority identity. Pair-specific n, spatial autocorrelation, correlated themes and outcome-estimation uncertainty limit comparisons. The 20 coefficients are descriptive; no p-values or significance claims are produced.\n\n### Data Table\n\nThe actual full 4 × 5 correlation matrix and n matrix appear in the generated-results section and the executed notebook. All 20 theme-condition pairs in this supplied snapshot have 2,956 valid counties.\n\n### Table Details — Column Definitions\n\n| Output / column | Type / unit | Definition |\n| --- | --- | --- |\n| Matrix `theme` | Text | Four official theme field identifiers; row labels are stored as a normal CSV column |\n| Matrix `DIABETES`, `CHD`, `BPHIGH`, `OBESITY`, `COPD` | Number, −1 to +1 | Unweighted pairwise-complete Spearman rho for adjusted prevalence |\n| Tidy `theme` | Text | `RPL_THEME1`–`RPL_THEME4` |\n| Tidy `disease` | Text | CDC measure identifier matching the corresponding `{disease}_adj` field |\n| Tidy `correlation` | Number, nullable | Spearman rho; missing for insufficient/constant pairs |\n| Tidy `n_counties` | Integer | Number of counties with both values present in that pair |\n| Count matrix disease columns | Integer | Same valid counts arranged as 4 × 5 |\n\n### Table Details — Calculations\n\nFor each theme–condition pair, let P contain counties with both fields nonmissing. Compute average ranks separately within P, then rho = PearsonCorr(rank(theme), rank(adjusted prevalence)). The implementation uses pandas ranking followed by Pearson correlation and therefore needs no extra SciPy dependency. Ties in already-ranked SVI values and rounded prevalence values are handled using average ranks. Fewer than three usable counties or a constant field gives an undefined coefficient, not zero; n remains recorded. This minimum is a practical guard for this workflow.\n\nThe population column is context only. Every eligible county has equal weight. The national correlations include NYC counties with complete pairs. Unmatched or partially missing counties remain in Q3 and can only enter the pairs they support.\n\n### Table Details — Checks\n\nExactly four rows and five disease columns in the coefficient matrix; exactly 20 unique theme–disease tidy records; identical matrix/tidy values; every n recomputed from the Q3 two-column nonmissing mask; all defined coefficients within −1 to +1. No inference about causality, individual risk, statistical significance, or ranking differences beyond descriptive associations.\n\n## Data validity and interpretation limits\n\n- **Years are different.** 2025 is the PLACES release label, not the observation year. CDC\'s 2025 release documentation assigns the five selected conditions to 2023 BRFSS-based estimates and uses 2023 county population / ACS 2019–2023 inputs. SVI 2022 uses ACS 2018–2022. This is a cross-period descriptive association, not a synchronized panel or temporal causal study. Uploaded file identities and definitions must still be confirmed against the original official download.\n- **Modeled outcomes.** PLACES uses multilevel regression and poststratification. These are estimates rather than direct county observations. Published 95% CIs are retained; the point-estimate Spearman calculations do not propagate the modeling uncertainty or correlation between estimates. A population count must not be treated as a survey sample size.\n- **Overlapping inputs.** CDC lists individual age, sex, race/ethnicity and education, county adult poverty below 150% of the federal poverty threshold, and geographic random effects. SVI Theme 1 includes poverty and education; Theme 2 includes age structure; Theme 3 includes racial/ethnic composition. Definitions and levels differ, but the information overlaps. Age adjustment does not remove all of this dependence. Theme 4 has less direct definitional overlap with these listed inputs; it is not guaranteed independent, unbiased or weaker/stronger.\n- **Ecological and spatial dependence.** A county association is not an individual-level relationship. Nearby counties share environments and may have dependent estimates, so treating counties as independent for significance testing would be questionable. This stage produces descriptive coefficients without p-values, spatial adjustment or causal claims.\n- **Missingness and geography.** Pairwise deletion changes the analysis sample. Missing states can affect how representative the national pattern is. Original missingness and cleaning-induced NA are audited separately. County-equivalent boundaries/vintages may differ; unmatched records remain visible.\n- **National percentile context.** SVI values are relative rankings, not disease probabilities or absolute deprivation measures. State-specific rankings cannot be concatenated as substitutes for national rankings. Do not recompute the national rankings after excluding missing outcome counties.\n\n## Inputs and execution\n\nPlace the two original CDC exports under `data/raw/` using these project filenames (renaming the file is acceptable; do not edit its contents):\n\n1. `places_2025_county_gis.csv`: [CDC PLACES 2025 County GIS-Friendly](https://data.cdc.gov/d/i46a-9kgh), [CSV export](https://data.cdc.gov/api/views/i46a-9kgh/rows.csv?accessType=DOWNLOAD). Needed fields: `CountyFIPS`, `StateAbbr`, `CountyName`, optional `TotalPop18plus`, and each of the five `{measure}_AdjPrev` / `{measure}_Adj95CI` pairs.\n2. `svi_2022_us_county.csv`: [CDC/ATSDR documentation page](https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html), [official data selector](https://svi.cdc.gov/dataDownloads/data-download.html). Select **2022 / United States / Counties / CSV**. Needed fields: `FIPS` (or `STCNTY`), `ST_ABBR`, `COUNTY`, `RPL_THEMES`, `RPL_THEME1`–`RPL_THEME4`. Do not upload the US tract file or a state-ranked export. The current documentation webpage may not expose a downloadable county CSV; if unavailable, request the original official export from the teammate rather than substitute synthetic or re-ranked data.\n\nIf available, add the existing teammate file as `data/processed/q1_county_prevalence.csv`. It is inspected and never overwritten. Its absence prevents exact verification of Q1 geographic scope even though the documented column conventions are aligned to Startup.\n\nOpen `notebooks/q3_q4_transformation.ipynb` in Google Colab and follow its upload/download instructions. No local Python installation is required. The notebook is self-contained, with ordinary pandas code and no external repo imports; Colab can upload original CSVs and export the completed project ZIP. If the official SVI selector supplies a direct county CSV link, set `SVI_CSV_URL` to that exact link rather than guessing an endpoint. For local use with pandas installed, from BioVis run `python src/q3_q4_transform.py --root . --no-download`.\n\n## Output contract and future integration\n\nSuccessful execution writes UTF-8 CSVs with no pandas index and empty fields for missing values:\n\n- `q3_county_svi.csv`: shared county table, five point estimates and ten CI columns, overall + four theme ranks, population, NYC flags and join status.\n- `q3_national_summary.csv`: five overall-SVI associations with n.\n- `q4_correlations.csv`: full 4 × 5 rho matrix plus a `theme` column.\n- `q4_correlations_tidy.csv`: all 20 theme / disease / correlation / n records.\n- `data/processed/` contains only the four final CSVs above. The 12-row presentation sample is selected directly from `q3_county_svi.csv`; Q4 sample sizes are read from `q4_correlations_tidy.csv`. Unmatched counties, missingness, cleaning issues, scope checks, raw schema, validation and provenance are retained in `docs/validation/`. No separate sample or counts CSV is exported.\n\nThe four primary CSVs support the planned views; validation records in `docs/validation/` preserve provenance without cluttering `data/processed/`. Outputs can be loaded through Observable `FileAttachment` in a future project. Use an explicit row parser for `county_fips` to preserve its five-character text form; parse numeric blanks as null rather than zero and parse the boolean field explicitly. Do not apply an automatic numeric type conversion to FIPS. No Observable Framework configuration, Python data loader or frontend dependency is included.\n\n## AI disclosure for the final README\n\nAI (OpenAI Codex/ChatGPT) assisted in reading the supplied project materials and writing/reviewing the Python/pandas transformation code, Colab notebook and preprocessing documentation. Empirical values must be computed from the original official CDC PLACES and CDC/ATSDR SVI files. No synthetic observations or fabricated correlations are used. See the current run status above and the source manifest to distinguish code preparation from completed data processing; source identity and results require human review. Future sketches/visualizations and the final report are outside this task.\n\n## Sources\n\n- [CDC PLACES county GIS-friendly 2025 dataset and dictionary](https://data.cdc.gov/d/i46a-9kgh).\n- [CDC PLACES 2025 current release notes](https://www.cdc.gov/places/current-release-notes/).\n- [CDC PLACES methodology](https://www.cdc.gov/places/methodology/).\n- [CDC PLACES FAQ: interpreting modeled estimates](https://www.cdc.gov/places/faqs/index.html).\n- [CDC/ATSDR SVI data and documentation](https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html).\n- [CDC SVI official selector: US vs state ranking universes](https://svi.cdc.gov/dataDownloads/data-download.html).\n- [SVI 2022 technical documentation](https://svi.cdc.gov/map/data/docs/SVI2022Documentation_5.17.2024.pdf), especially pp. 2–5 (missing values, national rankings, ACS period).\n\n## Actual run outputs\n\n<!-- GENERATED_RESULTS_START -->\n\nActual run completed at 2026-11-02T20:47:33.524736+00:00. These tables are generated from the cached raw CSVs.\n\n#### Q3 Data Table — fixed 12-county documentation sample\n\nNYC five counties plus seven named counties chosen for geographical/contextual breadth, not statistical representativeness. These sample rows are selected from `data/processed/q3_county_svi.csv`, which also contains adult population, all CIs, four themes and join status.\n\n| county_fips | state | county | DIABETES_adj | CHD_adj | BPHIGH_adj | OBESITY_adj | COPD_adj | RPL_THEMES | is_nyc | borough |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| 36005 | NY | Bronx | 14.8000 | 6.3000 | 34.6000 | 33.7000 | 6.4000 | 0.9965 | True | Bronx |\n| 36047 | NY | Kings | 11.1000 | 5.1000 | 28.8000 | 25.0000 | 5.4000 | 0.8874 | True | Brooklyn |\n| 36061 | NY | New York | 8.4000 | 4.4000 | 25.8000 | 19.5000 | 4.1000 | 0.7283 | True | Manhattan |\n| 36081 | NY | Queens | 12.2000 | 4.9000 | 29.8000 | 23.8000 | 4.6000 | 0.8024 | True | Queens |\n| 36085 | NY | Richmond | 10.1000 | 4.8000 | 30.3000 | 30.0000 | 5.2000 | 0.5956 | True | Staten Island |\n| 06037 | CA | Los Angeles | 11.6000 | 4.8000 | 28.4000 | 26.4000 | 4.5000 | 0.8629 | False | NA |\n| 17031 | IL | Cook | 10.7000 | 4.7000 | 28.9000 | 31.0000 | 5.2000 | 0.7964 | False | NA |\n| 48201 | TX | Harris | 13.2000 | 5.5000 | 32.9000 | 37.1000 | 5.5000 | 0.9201 | False | NA |\n| 04013 | AZ | Maricopa | 9.6000 | 5.0000 | 28.6000 | 30.6000 | 5.2000 | 0.7130 | False | NA |\n| 12086 | FL | Miami-Dade | 13.3000 | 5.7000 | 30.3000 | 29.3000 | 4.7000 | 0.8810 | False | NA |\n| 28049 | MS | Hinds | 15.3000 | 6.1000 | 45.7000 | 44.6000 | 7.0000 | 0.9154 | False | NA |\n| 01001 | AL | Autauga | 11.4000 | 5.6000 | 38.2000 | 39.5000 | 6.4000 | 0.2663 | False | NA |\n\nQ3 overall SVI Spearman correlations:\n\n| svi_measure | disease | correlation | n_counties |\n| --- | --- | --- | --- |\n| RPL_THEMES | DIABETES | 0.7514 | 2956 |\n| RPL_THEMES | CHD | 0.6188 | 2956 |\n| RPL_THEMES | BPHIGH | 0.6000 | 2956 |\n| RPL_THEMES | OBESITY | 0.3408 | 2956 |\n| RPL_THEMES | COPD | 0.4964 | 2956 |\n\n#### Q4 Data Table — Spearman correlation matrix\n\n| theme | DIABETES | CHD | BPHIGH | OBESITY | COPD |\n| --- | --- | --- | --- | --- | --- |\n| RPL_THEME1 | 0.7859 | 0.7088 | 0.6641 | 0.3753 | 0.6155 |\n| RPL_THEME2 | 0.5555 | 0.4764 | 0.4421 | 0.3321 | 0.3852 |\n| RPL_THEME3 | 0.5081 | 0.1626 | 0.3100 | 0.0691 | -0.0164 |\n| RPL_THEME4 | 0.4278 | 0.3545 | 0.3241 | 0.1679 | 0.2685 |\n\nPairwise valid county counts:\n\n| theme | DIABETES | CHD | BPHIGH | OBESITY | COPD |\n| --- | --- | --- | --- | --- | --- |\n| RPL_THEME1 | 2956 | 2956 | 2956 | 2956 | 2956 |\n| RPL_THEME2 | 2956 | 2956 | 2956 | 2956 | 2956 |\n| RPL_THEME3 | 2956 | 2956 | 2956 | 2956 | 2956 |\n| RPL_THEME4 | 2956 | 2956 | 2956 | 2956 | 2956 |\n\nQ4 complete tidy table:\n\n| theme | disease | correlation | n_counties |\n| --- | --- | --- | --- |\n| RPL_THEME1 | DIABETES | 0.7859 | 2956 |\n| RPL_THEME1 | CHD | 0.7088 | 2956 |\n| RPL_THEME1 | BPHIGH | 0.6641 | 2956 |\n| RPL_THEME1 | OBESITY | 0.3753 | 2956 |\n| RPL_THEME1 | COPD | 0.6155 | 2956 |\n| RPL_THEME2 | DIABETES | 0.5555 | 2956 |\n| RPL_THEME2 | CHD | 0.4764 | 2956 |\n| RPL_THEME2 | BPHIGH | 0.4421 | 2956 |\n| RPL_THEME2 | OBESITY | 0.3321 | 2956 |\n| RPL_THEME2 | COPD | 0.3852 | 2956 |\n| RPL_THEME3 | DIABETES | 0.5081 | 2956 |\n| RPL_THEME3 | CHD | 0.1626 | 2956 |\n| RPL_THEME3 | BPHIGH | 0.3100 | 2956 |\n| RPL_THEME3 | OBESITY | 0.0691 | 2956 |\n| RPL_THEME3 | COPD | -0.0164 | 2956 |\n| RPL_THEME4 | DIABETES | 0.4278 | 2956 |\n| RPL_THEME4 | CHD | 0.3545 | 2956 |\n| RPL_THEME4 | BPHIGH | 0.3241 | 2956 |\n| RPL_THEME4 | OBESITY | 0.1679 | 2956 |\n| RPL_THEME4 | COPD | 0.2685 | 2956 |\n\n#### Actual validation\n\n| metric | value |\n| --- | --- |\n| merged_counties | 3144.0000 |\n| matched_counties | 3143.0000 |\n| places_only_counties | 0.0000 |\n| svi_only_counties | 1.0000 |\n| fips_match_rate_places_denominator | 1.0000 |\n| fips_match_rate_svi_denominator | 0.9997 |\n| duplicate_fips_merged | 0.0000 |\n| explicit_scope_exclusions | 0.0000 |\n\nRaw source counts:\n\n| dataset | rows | duplicate_full_rows |\n| --- | --- | --- |\n| places | 3143 | 0 |\n| svi | 3144 | 0 |\n\nMissing values (non-NYC borough blanks are expected):\n\n| column | n_missing |\n| --- | --- |\n| county_fips | 0 |\n| state | 0 |\n| county | 0 |\n| pop18plus | 1 |\n| DIABETES_adj | 188 |\n| DIABETES_adj_lo | 188 |\n| DIABETES_adj_hi | 188 |\n| CHD_adj | 188 |\n| CHD_adj_lo | 188 |\n| CHD_adj_hi | 188 |\n| BPHIGH_adj | 188 |\n| BPHIGH_adj_lo | 188 |\n| BPHIGH_adj_hi | 188 |\n| OBESITY_adj | 188 |\n| OBESITY_adj_lo | 188 |\n| OBESITY_adj_hi | 188 |\n| COPD_adj | 188 |\n| COPD_adj_lo | 188 |\n| COPD_adj_hi | 188 |\n| RPL_THEMES | 0 |\n| RPL_THEME1 | 0 |\n| RPL_THEME2 | 0 |\n| RPL_THEME3 | 0 |\n| RPL_THEME4 | 0 |\n| is_nyc | 0 |\n| borough | 3139 |\n| join_status | 0 |\n\nQ1 alignment/reuse status: `{"exists": false, "reused": false, "scope_verified": false, "reason": "Q1 file was not provided; independently use official PLACES. Q1 scope/field convention comparison remains pending."}`\n\nFull checks, anomaly records, out-of-scope records, unmatched FIPS and source hashes are in `docs/validation/`.\n\n<!-- GENERATED_RESULTS_END -->\n'
STANDALONE_SCRIPT = '"""CDC county data transformation only; Python 3.10+, pandas 2+.\n\nRun from the repository root: python src/q3_q4_transform.py --root .\nThe notebook contains the same implementation for standalone Colab execution.\nNo plots, imputation, population weighting, or tract-rank aggregation.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport re\nimport sys\nimport urllib.request\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\n\nPLACES_ID = "i46a-9kgh"\nPLACES_URL = f"https://data.cdc.gov/api/views/{PLACES_ID}/rows.csv?accessType=DOWNLOAD"\nSVI_PORTAL = "https://svi.cdc.gov/dataDownloads/data-download.html"\nDISEASES = ["DIABETES", "CHD", "BPHIGH", "OBESITY", "COPD"]\nTHEMES = {\n    "RPL_THEME1": "Socioeconomic Status",\n    "RPL_THEME2": "Household Characteristics",\n    "RPL_THEME3": "Racial and Ethnic Minority Status",\n    "RPL_THEME4": "Housing Type and Transportation",\n}\nRANKS = ["RPL_THEMES", *THEMES]\nNYC = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan",\n       "36081": "Queens", "36085": "Staten Island"}\nSAMPLE_FIPS = [*NYC, "06037", "17031", "48201", "04013", "12086", "28049", "01001"]\nUS_STATES = set("01 02 04 05 06 08 09 10 11 12 13 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 44 45 46 47 48 49 50 51 53 54 55 56".split())\nMISSING_TOKENS = {"", "na", "n/a", "null", "none", "nan", "suppressed", "*", "**", "--", "-", "—"}\nRAW_NAMES = {"places": "places_2025_county_gis.csv", "svi": "svi_2022_us_county.csv"}\nVALUE_COLS = [f"{d}_adj" for d in DISEASES]\nHEALTH_COLS = [f"{d}_adj{s}" for d in DISEASES for s in ("", "_lo", "_hi")]\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef file_sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef read_raw(path):\n    """Read strings before cleaning so leading zeros and source missing tokens survive."""\n    df = pd.read_csv(path, dtype="string", keep_default_na=False, encoding="utf-8-sig")\n    original = list(df.columns)\n    df.columns = df.columns.str.strip()\n    if df.columns.duplicated().any():\n        raise ValueError(f"Duplicate column names after trimming in {path}: {original}")\n    if df.empty:\n        raise ValueError(f"No data rows in {path}")\n    return df\n\n\ndef col(df, *candidates, required=True):\n    lookup = {str(c).casefold(): c for c in df.columns}\n    for candidate in candidates:\n        if candidate.casefold() in lookup:\n            return lookup[candidate.casefold()]\n    if required:\n        raise ValueError(f"Missing required column {candidates}; actual columns: {list(df.columns)}")\n    return None\n\n\ndef inspect_frame(df, dataset):\n    """Report as-read dtypes separately from inferred numeric content."""\n    records = []\n    for name in df:\n        s = df[name].astype("string").str.strip()\n        numeric = pd.to_numeric(s, errors="coerce")\n        records.append({"dataset": dataset, "column": name, "read_dtype": str(df[name].dtype),\n                        "source_blank_count": int(s.eq("").sum()),\n                        "numeric_parseable_count": int(numeric.notna().sum()),\n                        "distinct_values": int(s.nunique(dropna=False))})\n    return pd.DataFrame(records)\n\n\ndef download_csv(url, dest):\n    """Atomically cache a complete CSV; never replace an existing raw snapshot."""\n    dest = Path(dest)\n    if dest.exists():\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    request = urllib.request.Request(url, headers={"User-Agent": "BioVis-academic-transformation/1.0"})\n    with urllib.request.urlopen(request, timeout=120) as response:\n        payload = response.read()\n        final_url = response.geturl()\n    # Parse before writing: CDC block pages must not become a raw-data cache.\n    import io\n    probe = pd.read_csv(io.BytesIO(payload), dtype="string", nrows=2)\n    if len(probe.columns) < 5 or probe.empty:\n        raise ValueError(f"The URL did not return a nonempty tabular CDC CSV: {url}")\n    temp = dest.with_suffix(".download")\n    temp.write_bytes(payload)\n    temp.replace(dest)\n    source_record = dest.parents[2] / "docs/validation/sources" / dest.with_suffix(".source.json").name\n    json_write(source_record, {\n        "requested_url": url, "resolved_url": final_url, "retrieved_utc": utc_now(),\n        "sha256": file_sha256(dest), "bytes": dest.stat().st_size,\n    })\n\n\ndef load_sources(root, download=True, svi_url=None):\n    """Inspect the optional teammate file first. Never modify it."""\n    root = Path(root)\n    q1_path = root / "data/processed/q1_county_prevalence.csv"\n    q1 = read_raw(q1_path) if q1_path.exists() else None\n    frames, provenance, failures = {}, {}, []\n    for dataset, filename in RAW_NAMES.items():\n        path = root / "data/raw" / filename\n        url = PLACES_URL if dataset == "places" else svi_url\n        if not path.exists() and download and url:\n            try:\n                download_csv(url, path)\n            except Exception as exc:\n                failures.append(f"{dataset} download: {type(exc).__name__}: {exc}")\n        if not path.exists():\n            failures.append(f"Missing {path}. " + (\n                f"Download CDC PLACES 2025 County GIS-Friendly ({PLACES_ID})." if dataset == "places" else\n                f"At {SVI_PORTAL} choose 2022 / United States / Counties / CSV; upload the national county file, not tracts or individual-state rankings."))\n            continue\n        frames[dataset] = read_raw(path)\n        sidecar = root / "docs/validation/sources" / path.with_suffix(".source.json").name\n        provenance[dataset] = {\n            "path": str(path.relative_to(root)), "sha256": file_sha256(path),\n            "bytes": path.stat().st_size, "rows": len(frames[dataset]),\n            "columns": list(frames[dataset].columns), "observed_utc": utc_now(),\n            "download_record": json.loads(sidecar.read_text()) if sidecar.exists() else None,\n            "source_identity": (f"Expected official PLACES 2025 county GIS CSV ({PLACES_ID})" if dataset == "places" else\n                                "Expected official SVI 2022 US county CSV, national rankings"),\n            "identity_note": "Schema checks alone cannot authenticate an uploaded file\'s year or ranking universe; retain the original CDC download and source record.",\n        }\n    if failures:\n        raise FileNotFoundError("\\n".join(failures))\n    inspections = [inspect_frame(df, name) for name, df in frames.items()]\n    if q1 is not None:\n        inspections.append(inspect_frame(q1, "q1_candidate"))\n        provenance["q1_candidate"] = {"path": str(q1_path.relative_to(root)), "sha256": file_sha256(q1_path), "rows": len(q1)}\n    raw_summary = pd.DataFrame([\n        {"dataset": name, "rows": len(df), "columns": len(df.columns),\n         "duplicate_full_rows": int(df.duplicated().sum())} for name, df in frames.items()\n    ])\n    return frames, q1, provenance, pd.concat(inspections, ignore_index=True), raw_summary\n\n\ndef normalize_fips(series, label):\n    s = series.astype("string").str.strip().str.replace(r"\\.0+$", "", regex=True)\n    bad = ~s.str.fullmatch(r"\\d{4,5}", na=False)\n    if bad.any():\n        examples = series[bad].head(10).tolist()\n        raise ValueError(f"{label}: invalid county FIPS ({int(bad.sum())} rows): {examples}. "\n                         "County identifiers must have 4 or 5 digits before zero-padding; 11-digit tracts are rejected.")\n    s = s.str.zfill(5)\n    if s.str.endswith("000").any():\n        raise ValueError(f"{label}: found state/national aggregate code ending in 000")\n    if s.duplicated().any():\n        raise ValueError(f"{label}: duplicate county FIPS; no automatic deduplication: {s[s.duplicated(False)].tolist()[:20]}")\n    return s\n\n\ndef text_values(s):\n    s = s.astype("string").str.strip()\n    return s.mask(s.str.casefold().isin(MISSING_TOKENS))\n\n\ndef numeric_clean(s, fips, dataset, field, issues, lo=0, hi=None, integer=False, record_missing=True):\n    raw = s.astype("string").str.strip()\n    missing = raw.isna() | raw.str.casefold().isin(MISSING_TOKENS)\n    numeric_text = raw.str.replace(r"(?<=\\d),(?=\\d{3}(?:\\D|$))", "", regex=True)\n    num = pd.to_numeric(numeric_text.mask(missing), errors="coerce").astype("Float64")\n    sentinel = num.eq(-999).fillna(False)\n    parse_error = ~missing & num.isna()\n    invalid = num.notna() & ~sentinel & (~num.ge(lo) | (num.gt(hi) if hi is not None else False))\n    if integer:\n        invalid = invalid | (num.notna() & ~sentinel & num.mod(1).ne(0))\n    for reason, mask in [("source_missing", missing), ("sentinel_-999", sentinel),\n                         ("numeric_parse_error", parse_error), ("out_of_range_or_noninteger", invalid)]:\n        if reason == "source_missing" and not record_missing:\n            continue\n        for idx in s.index[mask.fillna(False)]:\n            issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": field,\n                           "raw_value": None if pd.isna(raw.loc[idx]) else str(raw.loc[idx]), "reason": reason})\n    return num.mask(missing | sentinel | parse_error | invalid)\n\n\ndef parse_ci(s, fips, dataset, disease, issues):\n    raw = s.astype("string").str.strip()\n    # CDC GIS-friendly intervals are strings such as (8.2, 9.3).\n    pattern = r"^\\s*[\\[(]?\\s*(-?\\d+(?:\\.\\d+)?)\\s*[,–-]\\s*(-?\\d+(?:\\.\\d+)?)\\s*[\\])]?\\s*$"\n    extracted = raw.str.extract(pattern)\n    missing = raw.isna() | raw.str.casefold().isin(MISSING_TOKENS) | raw.isin(["-999", "-999.0"])\n    malformed = ~missing & extracted[0].isna()\n    for idx in raw.index[malformed]:\n        issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": f"{disease}_adj_ci",\n                       "raw_value": str(raw.loc[idx]), "reason": "malformed_ci"})\n    # Keep one record of the original interval for every missing/invalid CI.\n    for idx in raw.index[missing]:\n        issues.append({"dataset": dataset, "county_fips": str(fips.loc[idx]), "field": f"{disease}_adj_ci",\n                       "raw_value": None if pd.isna(raw.loc[idx]) else str(raw.loc[idx]), "reason": "source_missing_or_sentinel_ci"})\n    bounds = []\n    for i, suffix in enumerate(["lo", "hi"]):\n        bounds.append(numeric_clean(extracted[i], fips, dataset, f"{disease}_adj_{suffix}", issues, hi=100, record_missing=False))\n    return bounds\n\n\ndef clean_sources(frames):\n    issues, exclusions = [], []\n    p, s = frames["places"], frames["svi"]\n    pfips = normalize_fips(p[col(p, "CountyFIPS")], "PLACES")\n    sfips = normalize_fips(s[col(s, "FIPS", "STCNTY")], "SVI")\n    # If the file includes both, they must identify the same counties, never tracts.\n    if col(s, "STCNTY", required=False) and col(s, "FIPS", required=False):\n        stcnty = normalize_fips(s[col(s, "STCNTY")], "SVI STCNTY")\n        if not sfips.equals(stcnty):\n            raise ValueError("SVI FIPS and STCNTY disagree; confirm a county CSV was supplied")\n    if pfips.str[:2].nunique() < 40 or sfips.str[:2].nunique() < 40:\n        raise ValueError("Expected national county CSVs, but fewer than 40 state FIPS prefixes were found")\n    places = pd.DataFrame({"county_fips": pfips,\n                           "state": text_values(p[col(p, "StateAbbr")]),\n                           "county": text_values(p[col(p, "CountyName")])})\n    adult_col = col(p, "TotalPop18plus", required=False)\n    places["pop18plus"] = (numeric_clean(p[adult_col], pfips, "places", "pop18plus", issues, integer=True)\n                                   if adult_col else pd.Series(pd.NA, index=p.index, dtype="Float64"))\n    for disease in DISEASES:\n        places[f"{disease}_adj"] = numeric_clean(p[col(p, f"{disease}_AdjPrev")], pfips, "places", f"{disease}_adj", issues, hi=100)\n        lower, upper = parse_ci(p[col(p, f"{disease}_Adj95CI")], pfips, "places", disease, issues)\n        # Inconsistent uncertainty bounds are missing, without discarding the estimate/county.\n        point = places[f"{disease}_adj"]\n        bad_order = (lower.notna() & upper.notna() & lower.gt(upper))\n        outside = point.notna() & ((lower.notna() & point.lt(lower)) | (upper.notna() & point.gt(upper)))\n        bad = (bad_order | outside).fillna(False)\n        for idx in p.index[bad]:\n            issues.append({"dataset": "places", "county_fips": str(pfips.loc[idx]), "field": f"{disease}_adj_ci",\n                           "raw_value": str(p.loc[idx, col(p, f"{disease}_Adj95CI")]), "reason": "ci_order_or_estimate_outside_ci"})\n        places[f"{disease}_adj_lo"] = lower.mask(bad)\n        places[f"{disease}_adj_hi"] = upper.mask(bad)\n    svi = pd.DataFrame({"county_fips": sfips,\n                        "svi_state": text_values(s[col(s, "ST_ABBR")]),\n                        "svi_county": text_values(s[col(s, "COUNTY")])})\n    for rank in RANKS:\n        svi[rank] = numeric_clean(s[col(s, rank)], sfips, "svi", rank, issues, hi=1)\n    cleaned = {}\n    for label, df in [("places", places), ("svi", svi)]:\n        in_scope = df.county_fips.str[:2].isin(US_STATES)\n        for fips in df.loc[~in_scope, "county_fips"]:\n            exclusions.append({"dataset": label, "county_fips": fips, "reason": "Outside 50 states + DC; not in US national SVI ranking universe"})\n        cleaned[label] = df.loc[in_scope].copy()\n    return cleaned, pd.DataFrame(issues, columns=["dataset", "county_fips", "field", "raw_value", "reason"]), pd.DataFrame(exclusions, columns=["dataset", "county_fips", "reason"])\n\n\ndef check_q1(q1, places):\n    """Reuse only verified identical values; record scope differences without changing files."""\n    report = {"exists": q1 is not None, "reused": False, "scope_verified": False}\n    differences = pd.DataFrame(columns=["county_fips", "scope_status"])\n    if q1 is None:\n        report["reason"] = "Q1 file was not provided; independently use official PLACES. Q1 scope/field convention comparison remains pending."\n        return places, report, differences\n    try:\n        fips_col = col(q1, "county_fips", "CountyFIPS")\n        candidate = q1.copy()\n        candidate["county_fips"] = normalize_fips(q1[fips_col], "Q1")\n        qa, pa = set(candidate.county_fips), set(places.county_fips)\n        differences = pd.DataFrame([\n            *({"county_fips": f, "scope_status": "q1_only"} for f in sorted(qa - pa)),\n            *({"county_fips": f, "scope_status": "official_places_only"} for f in sorted(pa - qa)),\n        ], columns=["county_fips", "scope_status"])\n        report.update({"q1_counties": len(qa), "official_places_counties": len(pa),\n                       "q1_only_count": len(qa - pa), "official_places_only_count": len(pa - qa),\n                       "scope_verified": qa == pa})\n        missing = [c for c in HEALTH_COLS if c not in candidate]\n        report["missing_canonical_fields"] = missing\n        if missing or qa != pa:\n            report["reason"] = "Q1 fields/scope differ. Use official PLACES; all scope differences are exported. Team reconciliation is required before claiming Q1 alignment."\n            return places, report, differences\n        candidate = candidate.set_index("county_fips").sort_index()\n        official = places.set_index("county_fips").sort_index()\n        for c in HEALTH_COLS:\n            candidate[c] = pd.to_numeric(candidate[c], errors="coerce").astype("Float64")\n        pd.testing.assert_frame_equal(candidate[HEALTH_COLS], official[HEALTH_COLS],\n                                      check_dtype=False, check_exact=False, rtol=0, atol=1e-9)\n        official.loc[:, HEALTH_COLS] = candidate[HEALTH_COLS]\n        report.update({"reused": True, "reason": "Q1 point estimates and CIs match the official release and full geographic scope; verified values reused; source file untouched."})\n        return official.reset_index(), report, differences\n    except (ValueError, AssertionError) as exc:\n        report["reason"] = f"Q1 cannot be safely reused: {exc}"\n        return places, report, differences\n\n\ndef merge_counties(cleaned, q1=None):\n    places, q1_report, scope_diff = check_q1(q1, cleaned["places"])\n    merged = places.merge(cleaned["svi"], on="county_fips", how="outer", validate="one_to_one", indicator=True)\n    merged["join_status"] = merged.pop("_merge").astype("string").replace({"both": "matched", "left_only": "places_only", "right_only": "svi_only"})\n    merged["state"] = merged.state.combine_first(merged.svi_state)\n    merged["county"] = merged.county.combine_first(merged.svi_county)\n    merged["is_nyc"] = merged.county_fips.isin(NYC)\n    merged["borough"] = merged.county_fips.map(NYC).astype("string")\n    state_conflicts = merged.loc[merged.svi_state.notna() & merged.state.notna() & merged.svi_state.ne(merged.state),\n                                 ["county_fips", "state", "svi_state"]]\n    if not state_conflicts.empty:\n        raise ValueError(f"State abbreviations conflict across sources: {state_conflicts.to_dict(\'records\')}")\n    merged = merged.drop(columns=["svi_state", "svi_county"]).sort_values("county_fips").reset_index(drop=True)\n    return merged, q1_report, scope_diff\n\n\ndef correlations(df, predictors):\n    """Spearman rho = Pearson correlation of within-pair average ranks.\n\n    Rank after pairwise deletion; otherwise missingness changes ranks incorrectly.\n    No weights, imputation, p-values, or causal interpretations.\n    """\n    records = []\n    for theme in predictors:\n        for disease in DISEASES:\n            pair = df[[theme, f"{disease}_adj"]].dropna()\n            n = len(pair)\n            rho = None\n            if n >= 3 and pair.nunique().min() > 1:\n                value = pair.rank(method="average").astype(float).corr(method="pearson").iloc[0, 1]\n                rho = float(value) if math.isfinite(value) else None\n            records.append({"theme": theme, "disease": disease, "correlation": rho, "n_counties": n})\n    return pd.DataFrame(records)\n\n\ndef q3_transform(merged):\n    # Retain all 4 theme ranks here so Q4 uses exactly the same merged snapshot.\n    q3 = merged[["county_fips", "state", "county", "pop18plus", *HEALTH_COLS,\n                 *RANKS, "is_nyc", "borough", "join_status"]].copy()\n    summary = correlations(q3, ["RPL_THEMES"]).rename(columns={"theme": "svi_measure"})\n    sample = q3.set_index("county_fips").reindex(SAMPLE_FIPS)\n    missing_sample = sample["county"].isna()\n    if missing_sample.any():\n        raise ValueError(f"Fixed documentation sample counties absent or unnamed: {sample.index[missing_sample].tolist()}")\n    return q3, summary, sample.reset_index()\n\n\ndef q4_transform(q3):\n    tidy = correlations(q3, THEMES)\n    matrix = tidy.pivot(index="theme", columns="disease", values="correlation").reindex(index=THEMES, columns=DISEASES)\n    counts = tidy.pivot(index="theme", columns="disease", values="n_counties").reindex(index=THEMES, columns=DISEASES)\n    return matrix, tidy, counts\n\n\ndef validate(frames, cleaned, merged, q3, summary, matrix, tidy, issues, exclusions, q1_report):\n    checks = []\n    def check(name, passed, detail):\n        checks.append({"check": name, "passed": bool(passed), "detail": detail})\n    check("unique_merged_fips", not merged.county_fips.duplicated().any(), len(merged))\n    check("five_digit_fips", merged.county_fips.str.fullmatch(r"\\d{5}").all(), "5-digit strings")\n    check("national_states_only", merged.county_fips.str[:2].isin(US_STATES).all(), "50 states + DC")\n    check("all_source_counties_retained", set(merged.county_fips) == set(cleaned["places"].county_fips) | set(cleaned["svi"].county_fips), "Outer join; no county removed for missing values")\n    check("nyc_five_present", set(NYC).issubset(set(merged.county_fips)), sorted(NYC))\n    nyc = merged[merged.is_nyc]\n    check("nyc_all_sources_matched", len(nyc) == 5 and nyc.join_status.eq("matched").all(), "Five NYC FIPS must occur in both sources")\n    check("nyc_boroughs_only", q3.loc[q3.is_nyc, "borough"].notna().all() and q3.loc[~q3.is_nyc, "borough"].isna().all(), "Official county names retained; borough labels added separately")\n    population = q3.pop18plus.dropna()\n    check("adult_population_valid", (population.ge(0) & population.mod(1).eq(0)).all(), "Nullable nonnegative integer; context only")\n    for c in HEALTH_COLS + RANKS:\n        hi = 1 if c in RANKS else 100\n        check(f"range_{c}", q3[c].dropna().between(0, hi).all(), f"Nonmissing range [0, {hi}]")\n    for d in DISEASES:\n        a, lo, hi = (q3[f"{d}_adj{s}"] for s in ("", "_lo", "_hi"))\n        valid = ((lo.isna() | hi.isna() | lo.le(hi)) & (a.isna() | lo.isna() | a.ge(lo)) & (a.isna() | hi.isna() | a.le(hi)))\n        check(f"ci_order_{d}", valid.fillna(False).all(), "lower <= estimate <= upper when available")\n    check("q4_shape", matrix.shape == (4, 5) and len(tidy) == 20, list(matrix.shape))\n    check("q4_unique_pairs", not tidy.duplicated(["theme", "disease"]).any(), "20 distinct theme-disease pairs")\n    check("q4_matrix_tidy_agree", all(\n        (pd.isna(row.correlation) and pd.isna(matrix.loc[row.theme, row.disease])) or\n        (not pd.isna(row.correlation) and row.correlation == matrix.loc[row.theme, row.disease])\n        for row in tidy.itertuples()), "All 20 matrix cells agree with tidy rows")\n    check("q3_five_correlations", len(summary) == 5, len(summary))\n    combined = pd.concat([summary.rename(columns={"svi_measure": "theme"}), tidy], ignore_index=True)\n    for row in combined.itertuples():\n        expected = int(q3[[row.theme, f"{row.disease}_adj"]].notna().all(axis=1).sum())\n        check(f"pair_n_{row.theme}_{row.disease}", row.n_counties == expected, expected)\n    check("rho_range", combined.correlation.dropna().between(-1, 1).all(), "[-1, 1]")\n    matched = int(merged.join_status.eq("matched").sum())\n    info = {\n        "run_utc": utc_now(), "status": "completed" if all(c["passed"] for c in checks) else "validation_failed",\n        "raw": {k: {"rows": len(v), "duplicate_full_rows": int(v.duplicated().sum())} for k, v in frames.items()},\n        "in_scope": {k: len(v) for k, v in cleaned.items()},\n        "merged_counties": len(merged), "matched_counties": matched,\n        "places_only_counties": int(merged.join_status.eq("places_only").sum()),\n        "svi_only_counties": int(merged.join_status.eq("svi_only").sum()),\n        "fips_match_rate_places_denominator": matched / len(cleaned["places"]),\n        "fips_match_rate_svi_denominator": matched / len(cleaned["svi"]),\n        "fips_match_rate_union_denominator": matched / len(merged),\n        "duplicate_fips_after_cleaning": {k: int(v.county_fips.duplicated().sum()) for k, v in cleaned.items()},\n        "duplicate_fips_merged": int(merged.county_fips.duplicated().sum()),\n        "missing_counts": {c: int(n) for c, n in q3.isna().sum().items()},\n        "issue_counts": {str(k): int(v) for k, v in issues.reason.value_counts().items()},\n        "explicit_scope_exclusions": len(exclusions), "q1": q1_report,\n        "observed_ranges": {c: {"min": None if q3[c].dropna().empty else float(q3[c].min()),\n                                 "max": None if q3[c].dropna().empty else float(q3[c].max())} for c in HEALTH_COLS + RANKS},\n        "nyc": {f: {"present": bool(merged.county_fips.eq(f).any()), "borough": b} for f, b in NYC.items()},\n        "correlations": combined.astype(object).where(combined.notna(), None).to_dict("records"),\n        "checks": checks,\n    }\n    if not all(c["passed"] for c in checks):\n        failed = [c["check"] for c in checks if not c["passed"]]\n        raise ValueError(f"Validation failed; processed CSVs not exported: {failed}")\n    return info, pd.DataFrame(checks)\n\n\ndef markdown_table(df, digits=4):\n    # Avoid an optional tabulate dependency in Colab.\n    def fmt(v):\n        if pd.isna(v):\n            return "NA"\n        if isinstance(v, float):\n            return f"{v:.{digits}f}"\n        return str(v).replace("|", "\\\\|").replace("\\n", " ")\n    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "| " + " | ".join(["---"] * len(df.columns)) + " |"]\n    lines.extend("| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False, name=None))\n    return "\\n".join(lines)\n\n\ndef write_results_section(root, sample, summary, matrix, tidy, counts, audit):\n    notes = root / "q3_q4_transformation_notes.md"\n    if not notes.exists():\n        raise FileNotFoundError("Keep q3_q4_transformation_notes.md with the repository; it contains definitions and limitations.")\n    marker = "<!-- GENERATED_RESULTS_START -->"\n    endmarker = "<!-- GENERATED_RESULTS_END -->"\n    text = notes.read_text(encoding="utf-8")\n    if marker not in text or endmarker not in text:\n        raise ValueError("Notes results markers missing")\n    section = (\n        f"\\n\\nActual run completed at {audit[\'run_utc\']}. These tables are generated from the cached raw CSVs.\\n\\n"\n        "#### Q3 Data Table — fixed 12-county documentation sample\\n\\n"\n        "NYC five counties plus seven named counties chosen for geographical/contextual breadth, not statistical representativeness. "\n        "These rows are selected directly from data/processed/q3_county_svi.csv, which also contains adult population, all CIs, four themes and join status.\\n\\n"\n        + markdown_table(sample[["county_fips", "state", "county", *VALUE_COLS, "RPL_THEMES", "is_nyc", "borough"]])\n        + "\\n\\nQ3 overall SVI Spearman correlations:\\n\\n" + markdown_table(summary)\n        + "\\n\\n#### Q4 Data Table — Spearman correlation matrix\\n\\n" + markdown_table(matrix.reset_index())\n        + "\\n\\nPairwise valid county counts:\\n\\n" + markdown_table(counts.reset_index())\n        + "\\n\\nQ4 complete tidy table:\\n\\n" + markdown_table(tidy)\n        + "\\n\\n#### Actual validation\\n\\n"\n        + markdown_table(pd.DataFrame([\n            {"metric": key, "value": audit[key]} for key in ["merged_counties", "matched_counties", "places_only_counties", "svi_only_counties",\n             "fips_match_rate_places_denominator", "fips_match_rate_svi_denominator", "duplicate_fips_merged", "explicit_scope_exclusions"]]))\n        + "\\n\\nRaw source counts:\\n\\n" + markdown_table(pd.DataFrame([{"dataset": k, **v} for k, v in audit["raw"].items()]))\n        + "\\n\\nMissing values (non-NYC borough blanks are expected):\\n\\n"\n        + markdown_table(pd.DataFrame([{"column": k, "n_missing": v} for k, v in audit["missing_counts"].items()]))\n        + "\\n\\nQ1 alignment/reuse status: `" + json.dumps(audit["q1"], ensure_ascii=False).replace("`", "\'") + "`\\n\\n"\n        + "Full checks, anomaly records, out-of-scope records, unmatched FIPS and source hashes are in the exported audit files.\\n\\n"\n    )\n    before, rest = text.split(marker, 1)\n    _, after = rest.split(endmarker, 1)\n    result = before + marker + section + endmarker + after\n    status_start, status_end = "<!-- RUN_STATUS_START -->", "<!-- RUN_STATUS_END -->"\n    if status_start in result and status_end in result:\n        before_status, rest_status = result.split(status_start, 1)\n        _, after_status = rest_status.split(status_end, 1)\n        result = (before_status + status_start + f"\\n\\n**Current status: empirical transformation completed at {audit[\'run_utc\']}; implemented validation checks passed.**\\n\\n"\n                  "Actual computed results are in the generated-results section. Source hashes and download records are in the run manifest. "\n                  "An uploaded file\'s release year and national ranking universe still require confirmation against the original CDC export. "\n                  "Q1 alignment status is recorded separately and may remain unresolved.\\n\\n" + status_end + after_status)\n    notes.write_text(result, encoding="utf-8")\n\n\ndef export_results(root, q3, summary, sample, matrix, tidy, counts, audit, checks,\n                   issues, exclusions, scope_diff, inspection, raw_summary, provenance):\n    root = Path(root)\n    out = root / "data/processed"\n    out.mkdir(parents=True, exist_ok=True)\n    validation_dir = root / "docs/validation"\n    validation_dir.mkdir(parents=True, exist_ok=True)\n    outputs = {\n        "q3_county_svi.csv": q3, "q3_national_summary.csv": summary,\n        "q4_correlations.csv": matrix.reset_index(), "q4_correlations_tidy.csv": tidy,\n    }\n    diagnostics = {\n        "unmatched_counties.csv": q3.loc[q3.join_status.ne("matched"), ["county_fips", "state", "county", "join_status"]],\n        "missingness.csv": pd.DataFrame([{"column": c, "n_missing": n, "n_counties": len(q3)} for c, n in audit["missing_counts"].items()]),\n        "cleaning_issues.csv": issues, "scope_exclusions.csv": exclusions,\n        "q1_scope_differences.csv": scope_diff, "raw_schema.csv": inspection,\n        "raw_summary.csv": raw_summary, "validation_checks.csv": checks,\n    }\n    # Write notes first to catch missing documentation templates before CSV export.\n    write_results_section(root, sample, summary, matrix, tidy, counts, audit)\n    for filename, df in outputs.items():\n        df.to_csv(out / filename, index=False, encoding="utf-8", na_rep="", float_format="%.10g")\n    for filename, df in diagnostics.items():\n        df.to_csv(validation_dir / filename, index=False, encoding="utf-8", na_rep="", float_format="%.10g")\n    json_write(validation_dir / "validation.json", audit)\n    manifest = {"run_utc": audit["run_utc"], "status": audit["status"], "sources": provenance,\n                "software": {"python": sys.version.split()[0], "pandas": pd.__version__},\n                "parameters": {"scope": "Union of official county FIPS in 50 states + DC", "join": "outer one_to_one",\n                               "correlation": "Spearman; pairwise complete; average ranks for ties", "weighted": False,\n                               "places_release": 2025, "svi_release": 2022,\n                               "nyc_borough_map": NYC, "sample_fips": SAMPLE_FIPS},\n                "outputs": {name: {"rows": len(df), "sha256": file_sha256(out / name)} for name, df in outputs.items()},\n                "validation_files": {name: {"rows": len(df), "sha256": file_sha256(validation_dir / name)} for name, df in diagnostics.items()}}\n    json_write(validation_dir / "run_manifest.json", manifest)\n    (validation_dir / "last_attempt.json").unlink(missing_ok=True)\n    return list(outputs)\n\n\ndef run(root, download=True, svi_url=None):\n    root = Path(root).resolve()\n    frames, q1, provenance, inspection, raw_summary = load_sources(root, download, svi_url)\n    cleaned, issues, exclusions = clean_sources(frames)\n    merged, q1_report, scope_diff = merge_counties(cleaned, q1)\n    q3, summary, sample = q3_transform(merged)\n    matrix, tidy, counts = q4_transform(q3)\n    audit, checks = validate(frames, cleaned, merged, q3, summary, matrix, tidy, issues, exclusions, q1_report)\n    files = export_results(root, q3, summary, sample, matrix, tidy, counts, audit, checks,\n                           issues, exclusions, scope_diff, inspection, raw_summary, provenance)\n    return {"q3": q3, "summary": summary, "sample": sample, "matrix": matrix, "tidy": tidy,\n            "counts": counts, "audit": audit, "files": files}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parents[1])\n    parser.add_argument("--no-download", action="store_true", help="Use cached/uploaded CSVs only")\n    parser.add_argument("--svi-url", default=None, help="Exact official US county CSV URL obtained from the CDC download selector; no guessed endpoint")\n    args = parser.parse_args()\n    try:\n        result = run(args.root, not args.no_download, args.svi_url)\n    except Exception as exc:\n        failure = {"status": "blocked_or_failed", "observed_utc": utc_now(), "error_type": type(exc).__name__, "error": str(exc),\n                   "note": "No successful new data transformation is claimed. Existing outputs, if any, may belong to an earlier run."}\n        json_write(args.root / "docs/validation/last_attempt.json", failure)\n        print(json.dumps(failure, ensure_ascii=False, indent=2), file=sys.stderr)\n        return 1\n    print(result["sample"][["county_fips", "county", *VALUE_COLS, "RPL_THEMES"]].to_string(index=False))\n    print(result["matrix"].to_string())\n    print(json.dumps({k: result["audit"][k] for k in ["status", "merged_counties", "matched_counties", "places_only_counties", "svi_only_counties"]}, indent=2))\n    print("\\n".join(result["files"]))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
notes_path = PROJECT_ROOT / "q3_q4_transformation_notes.md"
if not notes_path.exists():
    notes_path.write_text(NOTES_TEMPLATE, encoding="utf-8")
script_path = PROJECT_ROOT / "src/q3_q4_transform.py"
if not script_path.exists():
    script_path.write_text(STANDALONE_SCRIPT, encoding="utf-8")

output_files = export_results(PROJECT_ROOT, q3, summary, sample, matrix, tidy, counts,
    audit, checks, issues, exclusions, scope_diff, inspection, raw_summary, provenance)
display(pd.DataFrame([{"file": "data/processed/" + name,
                       "bytes": (PROJECT_ROOT / "data/processed" / name).stat().st_size} for name in output_files]))
print("Notes:", notes_path.name)
print("Validation:", audit["status"])
print("Q1 alignment:", q1_report)

# Verify actual CSV round trips preserve FIPS and output layout.
roundtrip = pd.read_csv(PROJECT_ROOT / "data/processed/q3_county_svi.csv", dtype={"county_fips": "string"})
assert roundtrip.county_fips.equals(q3.county_fips.reset_index(drop=True))
assert not any(str(c).startswith("Unnamed:") for c in roundtrip)
saved_matrix = pd.read_csv(PROJECT_ROOT / "data/processed/q4_correlations.csv")
assert saved_matrix.shape == (4, 6) and list(saved_matrix.columns) == ["theme", *DISEASES]
print("CSV round trips: passed")


Notes: q3_q4_transformation_notes.md
Validation: completed
Q1 alignment: {'exists': False, 'reused': False, 'scope_verified': False, 'reason': 'Q1 file was not provided; independently use official PLACES. Q1 scope/field convention comparison remains pending.'}
CSV round trips: passed


,file,bytes
0,data/processed/q3_county_svi.csv,423358
1,data/processed/q3_national_summary.csv,220
2,data/processed/q4_correlations.csv,345
3,data/processed/q4_correlations_tidy.csv,751


In [28]:
DOWNLOAD_ZIP = False  # set True in Colab to download all generated project files
if DOWNLOAD_ZIP:
    if not IN_COLAB:
        raise RuntimeError("Use the local BioVis directory directly outside Colab")
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/BioVis", "zip", PROJECT_ROOT.parent, PROJECT_ROOT.name)
    files.download(archive)
print("Save/download this executed .ipynb from Colab, then commit it with the real CSVs and manifest to GitHub.")


Save/download this executed .ipynb from Colab, then commit it with the real CSVs and manifest to GitHub.
